# California Housing : exploration, analyse statistique et choix du modèle

## Contexte et objectif

Le but est de prédire la **valeur médiane des logements** (`MedHouseVal`) d'un district californien
à partir de 8 caractéristiques socio-démographiques et géographiques. Les données viennent du
recensement américain de 1990 et sont fournies par scikit-learn (`fetch_california_housing`). Je les
ai exportées dans `sample_data/california_housing.csv` avec `02_Data_Generation.ipynb`.

Une précision importante dès le départ : une ligne n'est **pas une maison** mais un **district**
(un *block group* du recensement, de quelques centaines à quelques milliers d'habitants). Toutes les
variables sont donc des moyennes ou des médianes à l'échelle d'un quartier.

## Pourquoi une valeur médiane par district, et pas le prix de chaque maison ?

Au quotidien, ce qui intéresse un acheteur, un vendeur ou un banquier, c'est le prix **d'une maison
précise**. Ce jeu de données ne répond pas à cette question, et il est important de comprendre
pourquoi avant d'interpréter le moindre résultat.

**1. La source impose l'échelle du district.** Le recensement américain ne publie jamais de
données individuelles, pour protéger la vie privée des ménages : il diffuse uniquement des
statistiques agrégées par zone. La valeur maison par maison n'existe tout simplement pas dans
cette source.

**2. La médiane décrit le logement typique du quartier, mieux que la moyenne.** Les prix
immobiliers sont très asymétriques. Dans un district de 300 logements valant 150 000 $, trois
villas à 2 millions de dollars suffisent à faire monter la moyenne à environ 170 000 $, alors que
la médiane reste à 150 000 $. La médiane est la valeur qui coupe le quartier en deux : la moitié
des logements vaut moins, l'autre moitié plus. Elle résiste aux quelques biens exceptionnels, et
c'est pour cette raison que les statistiques immobilières officielles (prix médian de vente,
loyer médian) utilisent presque toujours la médiane.

**3. Ce que cela change pour l'usage du modèle.** Le modèle estime le **niveau de prix d'un
quartier**, pas le prix d'un bien particulier. Deux maisons du même district peuvent valoir du
simple au double selon leur taille, leur état ou leur vue, et le modèle n'en sait rien. Il est donc
adapté pour **comparer et prioriser des zones** (où investir, quels quartiers sont sous-évalués,
où les logements sont inabordables par rapport aux revenus), et **pas** pour faire une offre
d'achat ou une expertise individuelle. Pour cela, il faudrait des données de transactions,
maison par maison.

Garder cette limite en tête évite le contresens le plus courant sur ce jeu de données : lire une
erreur de 50 000 $ comme « je me trompe de 50 000 $ sur le prix de votre maison », alors qu'elle
signifie « je me trompe de 50 000 $ sur la valeur typique de votre quartier ».

Ce notebook me sert à **choisir le prétraitement et le modèle** que j'industrialiserai ensuite dans
le pipeline Azure ML du repo (`components/`, `ml/pipelines/`). Chaque décision prise ici doit être
justifiée, pour pouvoir être reprise sans hésitation dans le code de production.

## À quoi sert le modèle

Le cas d'usage que je vise est une **estimation automatique de la valeur immobilière d'un
district** à partir de données de recensement, sans visite ni expertise sur place. Ce type
d'estimation sert par exemple à :
- repérer des zones **sous-évaluées ou surévaluées** par rapport à leur profil socio-économique
  (ciblage d'investissements, veille d'un marché) ;
- donner un **ordre de grandeur de la valeur des garanties** d'un portefeuille de prêts
  immobiliers, district par district ;
- alimenter des études d'aménagement ou de politique du logement (où les logements sont-ils
  inabordables par rapport aux revenus ?).

Dans tous ces cas, le modèle sert à **trier et prioriser des zones**, pas à fixer le prix d'une
maison précise : une erreur de quelques dizaines de milliers de dollars reste acceptable si elle
ne change pas le classement des districts.

## Comment je mesure la performance, et ce que les chiffres veulent dire

La cible est exprimée en **centaines de milliers de dollars** (1,8 = 180 000 $). La valeur médiane
typique d'un district est d'environ **180 000 $** (en dollars de 1990). Je traduis donc
systématiquement chaque métrique en dollars pour la rendre parlante :

| Métrique | Ce qu'elle mesure concrètement | Comment je la lis |
|---|---|---|
| **RMSE** (*Root Mean Squared Error*, racine de l'erreur quadratique moyenne) — métrique principale | Écart « typique » entre la valeur estimée et la vraie valeur, en pénalisant fortement les grosses erreurs | RMSE = 0,50 → l'estimation d'un district s'écarte typiquement de **±50 000 $**, et les grosses erreurs pèsent lourd |
| **MAE** (*Mean Absolute Error*, erreur absolue moyenne) | Erreur moyenne en valeur absolue, chaque erreur comptant pour sa taille réelle | MAE = 0,33 → en moyenne, je me trompe de **33 000 $** par district |
| **MAPE** (*Mean Absolute Percentage Error*, erreur absolue moyenne en pourcentage) | Erreur moyenne **en pourcentage** de la vraie valeur | MAPE = 18 % → l'estimation est en moyenne à 18 % de la réalité, quel que soit le niveau de prix |
| **R²** (coefficient de détermination) | Part des différences de valeur entre districts que le modèle explique | R² = 0,80 → le modèle explique 80 % des écarts de valeur entre districts ; les 20 % restants viennent de facteurs absents des données (état des logements, vue, écoles…) |

Je choisis la **RMSE** comme critère principal parce qu'en estimation immobilière, une grosse
erreur ponctuelle (surévaluer un district de 150 000 $) est bien plus grave que plusieurs petites :
c'est elle qui fausse une décision d'investissement ou la valeur d'une garantie. La MAE et la MAPE
servent à **communiquer** le résultat à un interlocuteur métier, parce qu'elles se lisent
directement (« en moyenne, on se trompe de X $, soit Y % »).

**Mon seuil d'acceptabilité** : pour un outil de tri de zones, je vise une erreur relative moyenne
(MAPE) **inférieure à 20 %** et une RMSE **inférieure à 0,50 (50 000 $)**. C'est aussi ce seuil
que j'utiliserai dans le pipeline pour décider si un modèle mérite d'être enregistré.

## Ma démarche

J'ai suivi le même ordre que sur mes autres projets de régression : **comprendre avant de
modéliser**. Le notebook est organisé en quatre phases, et les conclusions de chaque phase
alimentent la suivante.

| Phase | Section | Étape | Question à laquelle je réponds |
|---|---|---|---|
| **0. Cadrage** | 1 | Configuration | Environnement de travail et suivi des expériences (MLflow) |
| **1. EDA** (*Exploratory Data Analysis*, analyse exploratoire des données) — **analyse de forme** | 2 | Structure des données | Taille, types, valeurs manquantes, doublons, valeurs plafonnées ? |
| **1. EDA — analyse de fond** | 3 | Variable cible | Comment se distribue la valeur des logements, faut-il la transformer ? |
| | 4 | Analyse univariée | Comment se comporte chaque variable prise seule ? |
| | 5 | Analyse bivariée | Quelles variables influencent vraiment la valeur, et de quelle façon ? |
| | 6 | Anomalies | Y a-t-il des valeurs impossibles, plafonnées ou aberrantes ? |
| | 7 | Multicolinéarité | Quelles variables disent la même chose ? |
| | 8 | Modèle statistique (OLS, *Ordinary Least Squares* : régression linéaire par moindres carrés ordinaires) | Un modèle linéaire simple tient-il ses hypothèses ? |
| **2. Préparation des données** | 9 | Feature engineering et choix de prétraitement | Quelles transformations améliorent vraiment la prédiction ? |
| **3. Modélisation** | 10 | Comparaison des modèles | Quels algorithmes marchent le mieux, à protocole égal ? |
| | 11 | Optimisation | Combien gagne-t-on en réglant les meilleurs ? |
| **4. Évaluation et conclusion** | 12 | Évaluation finale | Quelle performance attendre sur des données jamais vues ? |
| | 13 | Conclusion | Que retenir, et comment passer au pipeline ? |

- L'**analyse de forme** décrit le contenant : combien de lignes et de colonnes, de quels types,
  avec quels trous et quels plafonds. Elle ne dit encore rien du lien avec la cible.
- L'**analyse de fond** étudie le contenu : distributions, relations avec la cible, anomalies,
  redondances entre variables, et un premier modèle statistique qui teste les hypothèses.
- La **préparation** transforme ces constats en décisions de prétraitement, chacune vérifiée par
  une expérience.

Le feature engineering est intégré **dans** le pipeline scikit-learn : le modèle final prend en
entrée les 8 colonnes brutes, exactement comme le composant `train` du repo les recevra.

# Phase 0 — Cadrage

Je prépare l'environnement de travail et le suivi des expériences avant de toucher aux données.

### 1. Configuration

#### 1.1 Librairies

Je regroupe tous les imports ici pour voir d'un coup d'œil les dépendances du notebook. C'est aussi
la liste de ce qu'il faudra déclarer dans l'environnement Azure ML (`ml/environments/conda.yml`) si
j'utilise ces outils en production.

In [ ]:
# --- Standard ---
import json
import os
import tempfile
import time
import warnings
from pathlib import Path

# --- Manipulation de données et visualisation ---
import matplotlib.pyplot as plt

# --- Suivi des expériences ---
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns

# --- Statistiques (tests, régression OLS, VIF) ---
import statsmodels.api as sm
from dotenv import load_dotenv

# --- Machine learning ---
from lightgbm import LGBMRegressor
from mlflow.models import infer_signature
from scipy import stats
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    IsolationForest,
    RandomForestRegressor,
    VotingRegressor,
)
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import (
    GroupKFold,
    KFold,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer
from sklearn.svm import SVR
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from xgboost import XGBRegressor

# Je masque les avertissements de dépréciation des librairies : ils n'affectent pas les résultats
# et noient la lecture. Je garde en revanche les RuntimeWarning (calculs numériques).
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 90

# Graine fixe partout : je veux pouvoir relancer le notebook et retrouver les mêmes chiffres.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

#### 1.2 Chemins et paramètres

Je centralise ici tout ce qui peut changer d'un projet à l'autre ou d'une exécution à l'autre.
C'est la première cellule à adapter sur un autre jeu de données.

`N_ITER_SEARCH` règle la durée de l'optimisation en section 11 (20 combinaisons testées × 5 plis par
modèle). Je le baisse à 5 quand je veux juste vérifier que tout tourne.

In [ ]:
# Le notebook peut être lancé depuis notebooks/ (VS Code, Jupyter) ou depuis la racine (nbconvert).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = PROJECT_ROOT / "sample_data" / "california_housing.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"  # dossier ignoré par git (sorties locales)
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "MedHouseVal"
TARGET_UNIT_USD = 100_000  # la cible est exprimée en centaines de milliers de dollars
N_FOLDS = 5  # 5 plis : bon compromis biais / variance / temps de calcul
N_ITER_SEARCH = 20  # combinaisons d'hyperparamètres testées par modèle en section 11
TOP_K_TO_TUNE = 3  # nombre de modèles que j'optimise (les meilleurs de la section 10)

# Le fichier .env (non versionné) contient mes identifiants Azure.
load_dotenv(PROJECT_ROOT / ".env")
print("Racine du projet :", PROJECT_ROOT)

#### 1.3 Suivi des expériences avec MLflow sur Azure ML

Je trace chaque modèle testé dans MLflow : paramètres, métriques, graphiques et modèle entraîné.
Sans ça, au bout de quelques dizaines d'essais, je ne sais plus quel réglage a donné quel score.

Les runs partent **dans mon workspace Azure ML** (onglet *Jobs* du studio) si ces variables sont
présentes dans `.env` :

```
AZURE_SUBSCRIPTION_ID=<id de la souscription>
AZURE_RESOURCE_GROUP=<resource group du workspace>
AZUREML_WORKSPACE_NAME=<nom du workspace>
MLFLOW_BACKEND=auto        # auto | azure | local
```

L'authentification passe par `DefaultAzureCredential`, qui s'appuie sur ma session `az login` :
aucun secret n'est écrit dans le code. Sans ces variables (ou avec `MLFLOW_BACKEND=local`), je
stocke les runs en local dans `mlflow.db`.

In [ ]:
EXPERIMENT_NAME = "california-housing-model-selection"


def configure_mlflow() -> str:
    # Choisit où envoyer les runs MLflow et renvoie un libellé lisible du mode utilisé.
    # Ordre de priorité : Azure ML (si configuré) > MLFLOW_TRACKING_URI > base SQLite locale.
    backend = os.getenv("MLFLOW_BACKEND", "auto").lower()
    azure_vars = [
        os.getenv(v)
        for v in ("AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZUREML_WORKSPACE_NAME")
    ]

    if backend == "azure" or (backend == "auto" and all(azure_vars)):
        # Import local : ces librairies ne sont nécessaires que si j'utilise Azure.
        from azure.ai.ml import MLClient
        from azure.identity import DefaultAzureCredential

        subscription_id, resource_group, workspace_name = azure_vars
        ml_client = MLClient(
            DefaultAzureCredential(), subscription_id, resource_group, workspace_name
        )
        # Chaque workspace Azure ML expose son propre serveur MLflow : je récupère son URI.
        uri = ml_client.workspaces.get(workspace_name).mlflow_tracking_uri
        mode = f"Azure ML (workspace '{workspace_name}')"
    elif os.getenv("MLFLOW_TRACKING_URI"):
        uri = os.environ["MLFLOW_TRACKING_URI"]
        mode = "MLFLOW_TRACKING_URI"
    else:
        uri = f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}"
        mode = "local (mlflow.db)"

    mlflow.set_tracking_uri(uri)
    mlflow.set_experiment(EXPERIMENT_NAME)
    # Je désactive l'autolog : il journaliserait chaque fit interne de la validation croisée et
    # de la recherche d'hyperparamètres. Je journalise moi-même ce qui compte.
    mlflow.sklearn.autolog(disable=True)
    return mode


print("Suivi MLflow :", configure_mlflow())

# Phase 1 — EDA (analyse exploratoire des données)

## Analyse de forme

Je décris d'abord le **contenant** : structure, types, qualité apparente et limites de
collecte des données. Aucune relation avec la cible n'est étudiée à ce stade.

### 2. Structure des données

Avant de chercher des relations, je veux savoir **ce que j'ai entre les mains** : combien de lignes
et de colonnes, quels types, des doublons, des valeurs manquantes, des valeurs plafonnées.

#### 2.1 Chargement et premières vérifications

In [ ]:
df_raw = pd.read_csv(DATA_FILE)
FEATURES = [c for c in df_raw.columns if c != TARGET]

print(f"{df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes ({len(FEATURES)} variables + cible)")
print("Types :", df_raw.dtypes.value_counts().to_dict())
print("Valeurs manquantes :", int(df_raw.isna().sum().sum()))
print("Doublons :", int(df_raw.duplicated().sum()))
df_raw.head()

Pas de valeur manquante, pas de doublon, et uniquement des variables numériques : ce jeu de données
est beaucoup plus « propre » en apparence que la plupart des données réelles. Je vais voir que
les difficultés sont ailleurs (plafonds, ratios extrêmes, dimension géographique).

#### 2.2 Dictionnaire des variables

D'après la documentation de scikit-learn, chaque variable décrit un district :

| Variable | Signification | Unité | Nature |
|---|---|---|---|
| `MedInc` | Revenu médian des ménages | dizaines de milliers de $ | Continue |
| `HouseAge` | Âge médian des logements | années | Discrète (entiers) |
| `AveRooms` | Nombre moyen de pièces par ménage | pièces | Ratio |
| `AveBedrms` | Nombre moyen de chambres par ménage | chambres | Ratio |
| `Population` | Nombre d'habitants du district | personnes | Comptage |
| `AveOccup` | Nombre moyen d'occupants par ménage | personnes | Ratio |
| `Latitude` | Latitude du centre du district | degrés | Géographique |
| `Longitude` | Longitude du centre du district | degrés | Géographique |
| **`MedHouseVal`** | **Valeur médiane des logements (cible)** | **centaines de milliers de $** | Continue |

Trois variables sont des **ratios** (divisées par le nombre de ménages). Elles deviennent très
instables dans les petits districts : quelques ménages atypiques suffisent à produire des moyennes
aberrantes. J'y reviens en section 6.

#### 2.3 Valeurs plafonnées (censure)

Les données du recensement ont été **plafonnées** à la collecte. Je le vérifie en comptant les
lignes qui atteignent exactement le maximum de chaque variable.

In [ ]:
censure = pd.DataFrame(
    {
        "maximum": df_raw.max(),
        "n_lignes_au_maximum": (df_raw == df_raw.max()).sum(),
        "% au maximum": (df_raw == df_raw.max()).mean() * 100,
    }
).sort_values("n_lignes_au_maximum", ascending=False)
censure

**Ce que j'en retiens :**
- **La cible est plafonnée à 5,00001**, soit 500 000 $. Près de 5 % des districts (965) sont à
  cette valeur exacte : leur vraie valeur médiane est « 500 000 $ **ou plus** », on ne sait pas
  de combien. Un modèle ne pourra jamais apprendre à prédire au-delà, et il aura du mal sur ces
  districts.
- **`HouseAge` est plafonné à 52 ans** (environ 6 % des districts) : « 52 » veut dire « 52 ans ou
  plus ». `MedInc` est aussi plafonné à 15 (150 000 $), mais sur très peu de lignes.
- Ces plafonds sont une **limite des données**, pas une erreur à corriger : je ne peux pas inventer
  la vraie valeur. Je garde ces lignes (en production, des districts chers existeront aussi) et je
  testerai en section 9 s'il est utile de **borner les prédictions** à l'intervalle observé.

## Analyse de fond

J'étudie maintenant le **contenu** : la cible, chaque variable, leurs relations, les
anomalies, les redondances, et enfin un modèle statistique de référence (sections 3 à 8).

### 3. La variable cible : `MedHouseVal`

J'étudie la cible avant les variables explicatives, parce que sa distribution conditionne le choix
de la métrique, du modèle et d'une éventuelle transformation.

In [ ]:
y_raw = df_raw[TARGET]
y_log = np.log1p(y_raw)  # log1p(x) = log(1 + x)
TARGET_MIN, TARGET_MAX = float(y_raw.min()), float(y_raw.max())  # bornes observées (censure)

print(y_raw.describe().to_string())
print(f"\nAsymétrie (skew)  : brute = {y_raw.skew():.3f} | log = {y_log.skew():.3f}")
print(f"Aplatissement     : brute = {y_raw.kurt():.3f} | log = {y_log.kurt():.3f}")

# Même trio de graphiques pour la cible brute et son log, pour comparer d'un coup d'œil.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, (serie, label) in enumerate([(y_raw, "MedHouseVal"), (y_log, "log1p(MedHouseVal)")]):
    sns.histplot(serie, bins=60, kde=True, ax=axes[row, 0])
    axes[row, 0].axvline(serie.max(), color="crimson", ls="--", label="plafond")
    axes[row, 0].legend()
    axes[row, 0].set_title(f"Distribution — {label}")
    sns.boxplot(x=serie, ax=axes[row, 1])
    axes[row, 1].set_title(f"Boîte à moustaches — {label}")
    stats.probplot(serie, dist="norm", plot=axes[row, 2])
    axes[row, 2].set_title(f"Q-Q plot — {label}")
plt.tight_layout()
plt.show()

Je complète la lecture visuelle par trois **tests de normalité** (H0 : la distribution est normale,
seuil de 5 %) :
- **Shapiro-Wilk** : le plus puissant, mais limité à 5 000 observations (au-delà, sa p-value n'est
  plus fiable) : je l'applique donc sur un échantillon aléatoire de 5 000 districts ;
- **D'Agostino K²** : fondé sur l'asymétrie et l'aplatissement ;
- **Jarque-Bera** : même principe, très utilisé sur les résidus de régression.

Avec plus de 20 000 observations, ces tests rejettent la normalité au moindre écart : je ne les lis
jamais seuls, mais avec le Q-Q plot et la valeur du skew.

In [ ]:
def normality_tests(serie: pd.Series) -> dict:
    # p-values des trois tests + asymétrie et aplatissement, pour un tableau de synthèse.
    sample = serie.sample(n=min(len(serie), 5000), random_state=RANDOM_STATE)
    return {
        "Shapiro-Wilk p (n=5000)": stats.shapiro(sample).pvalue,
        "D'Agostino K² p": stats.normaltest(serie).pvalue,
        "Jarque-Bera p": stats.jarque_bera(serie).pvalue,
        "skew": serie.skew(),
        "kurtosis": serie.kurt(),
    }


pd.DataFrame(
    {"MedHouseVal": normality_tests(y_raw), "log1p(MedHouseVal)": normality_tests(y_log)}
).T

**Mes conclusions sur la cible :**
- La cible brute est **asymétrique à droite** (skew ≈ 1) avec un **pic artificiel au plafond** de
  5,0. Le passage au log réduit l'asymétrie (skew ≈ 0,3) mais ne supprime évidemment pas le pic.
- Les tests rejettent la normalité dans les deux cas. Le Q-Q plot montre que l'écart vient surtout
  des deux extrémités : le plafond à droite, et quelques districts très bon marché à gauche.
- **Ce n'est pas une décision évidente.** Le log rend la distribution plus régulière, mais ma
  métrique (RMSE) est calculée sur l'échelle brute, et un modèle entraîné sur le log optimise une
  erreur relative, pas absolue. Plutôt que de trancher sur des arguments théoriques, je **compare les
  deux options en validation croisée** (CV, *cross-validation* : le jeu d'entraînement
  est découpé en 5 parts, et chaque part sert tour à tour à évaluer un modèle entraîné
  sur les 4 autres) en section 9.

### 4. Analyse univariée : les variables une par une

Je cherche les variables **asymétriques** (gênantes pour les modèles linéaires) et les **valeurs
extrêmes** (qui peuvent peser trop lourd dans un modèle).

In [ ]:
num_summary = pd.DataFrame(
    {
        "moyenne": df_raw[FEATURES].mean(),
        "médiane": df_raw[FEATURES].median(),
        "écart-type": df_raw[FEATURES].std(),
        "min": df_raw[FEATURES].min(),
        "p1": df_raw[FEATURES].quantile(0.01),
        "p99": df_raw[FEATURES].quantile(0.99),
        "max": df_raw[FEATURES].max(),
        "skew": df_raw[FEATURES].skew(),
    }
).sort_values("skew", ascending=False)
num_summary

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.flat, FEATURES, strict=True):
    sns.histplot(df_raw[col], bins=60, ax=ax)
    ax.set_title(f"{col} (skew={df_raw[col].skew():.1f})", fontsize=10)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Les ratios ont des queues si longues qu'on ne voit rien en échelle linéaire :
# je les affiche en échelle logarithmique pour distinguer le coeur de la distribution des extrêmes.
RATIOS = ["AveRooms", "AveBedrms", "AveOccup", "Population"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, RATIOS, strict=True):
    sns.boxplot(x=df_raw[col], ax=ax, log_scale=True)
    ax.set_title(f"{col} (échelle log)")
plt.tight_layout()
plt.show()

**Ce que j'en retiens :**
- **`MedInc`** est modérément asymétrique (skew ≈ 1,6), ce qui est classique pour un revenu.
- **Les ratios (`AveRooms`, `AveBedrms`, `AveOccup`) ont des queues extrêmes** : le 99e centile de
  `AveOccup` est d'environ 5,4 occupants par ménage, mais le maximum dépasse 1 200. Le coeur des
  distributions est pourtant parfaitement raisonnable. Ces extrêmes sont traités en section 6.
- **`Population`** est très asymétrique aussi (quelques districts de plus de 10 000 habitants).
- **`HouseAge`** est quasi symétrique, mais avec un pic au plafond de 52 ans.
- **`Latitude` et `Longitude`** ont une distribution **bimodale** : les deux pics correspondent aux
  deux grandes agglomérations, la baie de San Francisco (≈ 37,8°N) et Los Angeles (≈ 34°N).

Pour les modèles linéaires, j'appliquerai une transformation de **Yeo-Johnson**
(`PowerTransformer`) dans le pipeline : elle corrige l'asymétrie de chaque variable en choisissant
elle-même la force de la transformation, et elle réduit l'influence des valeurs extrêmes.

### 5. Analyse bivariée : quelles variables influencent la valeur ?

Toutes les variables sont numériques. Je croise donc chacune avec la cible de quatre façons
complémentaires :
- **corrélations de Pearson et de Spearman** : force d'une relation linéaire ou monotone ;
- **carte géographique** : la localisation ne se résume pas à une corrélation ;
- **analyse par déciles** (Kruskal-Wallis et η²) : capte aussi les relations **non monotones** ;
- **information mutuelle** : dépendance de toute forme.

Une variable qui ressort partout est un signal fiable.

#### 5.1 Corrélations avec la cible

- **Pearson** mesure une relation **linéaire** et est sensible aux valeurs extrêmes.
- **Spearman** travaille sur les rangs : il détecte toute relation **monotone** et résiste aux
  extrêmes. C'est lui que j'utilise pour trier, vu les queues des ratios.

In [ ]:
rows = []
for col in FEATURES:
    r_p, p_p = stats.pearsonr(df_raw[col], y_raw)
    r_s, p_s = stats.spearmanr(df_raw[col], y_raw)
    rows.append(
        {"variable": col, "pearson": r_p, "p_pearson": p_p, "spearman": r_s, "p_spearman": p_s}
    )
corr_target = (
    pd.DataFrame(rows).set_index("variable").sort_values("spearman", key=abs, ascending=False)
)
# Correction de Bonferroni : je teste 8 variables, donc je divise le seuil de 5 % par 8
# pour ne pas déclarer significatives des corrélations dues au hasard.
corr_target["significatif (Bonferroni)"] = corr_target["p_spearman"] < 0.05 / len(corr_target)
corr_target

In [ ]:
# Nuages de points sur un échantillon (20 000 points se superposent et deviennent illisibles),
# avec une courbe LOWESS (Locally Weighted Scatterplot Smoothing : régression locale)
# qui montre la forme réelle de la relation.
sample = df_raw.sample(4000, random_state=RANDOM_STATE)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, corr_target.index, strict=True):
    sns.regplot(
        x=sample[col],
        y=sample[TARGET],
        ax=ax,
        lowess=True,
        scatter_kws={"alpha": 0.15, "s": 8},
        line_kws={"color": "crimson"},
    )
    ax.set_title(f"{col} — ρ Spearman = {corr_target.loc[col, 'spearman']:.2f}", fontsize=10)
    if col in RATIOS:
        ax.set_xscale("log")
plt.tight_layout()
plt.show()

**Ce que j'en retiens :**
- **`MedInc` domine très nettement** (ρ ≈ 0,68) : le revenu des habitants est de loin le meilleur
  indicateur de la valeur des logements, avec une relation presque linéaire. On voit aussi la ligne
  horizontale du plafond à 5,0.
- **`AveOccup`** a une relation négative (ρ ≈ −0,26) : plus il y a d'occupants par logement, plus
  le district est modeste. **`AveRooms`** a une relation positive modérée.
- **`Latitude` et `Longitude`** ont des corrélations faibles, mais leurs courbes LOWESS ne sont pas
  du tout monotones. Une corrélation faible ne veut pas dire « peu d'effet » : elle veut dire « pas
  d'effet **linéaire** ». La carte suivante le montre clairement.
- **`Population`** n'a aucun lien visible avec la valeur (ρ ≈ 0).

#### 5.2 La dimension géographique

En immobilier, la localisation est souvent déterminante. Je place chaque district sur une carte
avec sa valeur médiane.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
sc = axes[0].scatter(
    df_raw["Longitude"],
    df_raw["Latitude"],
    c=df_raw[TARGET],
    cmap="viridis",
    s=df_raw["Population"] / 400,
    alpha=0.5,
)
fig.colorbar(sc, ax=axes[0], label="MedHouseVal (100 k$)")
axes[0].set(
    title="Valeur médiane par district (taille = population)", xlabel="Longitude", ylabel="Latitude"
)
# Repères : les grandes villes, qui serviront au feature engineering (section 9).
CITIES = {
    "San Francisco": (37.7749, -122.4194),
    "Los Angeles": (34.0522, -118.2437),
    "San Diego": (32.7157, -117.1611),
    "San Jose": (37.3382, -121.8863),
    "Sacramento": (38.5816, -121.4944),
}
for city, (lat, lon) in CITIES.items():
    axes[0].annotate(city, (lon, lat), fontsize=9, color="crimson", weight="bold")
    axes[0].plot(lon, lat, "r*", markersize=10)
hb = axes[1].hexbin(
    df_raw["Longitude"],
    df_raw["Latitude"],
    C=df_raw[TARGET],
    reduce_C_function=np.median,
    gridsize=45,
    cmap="viridis",
)
fig.colorbar(hb, ax=axes[1], label="médiane de MedHouseVal")
axes[1].set(title="Valeur médiane par zone (hexagones)", xlabel="Longitude")
plt.tight_layout()
plt.show()

**Ce que j'en retiens :** la carte montre ce que les corrélations ne voyaient pas. Les valeurs les
plus élevées se concentrent **sur la côte**, autour de la **baie de San Francisco** et de **Los
Angeles / San Diego**. L'intérieur (vallée centrale, désert) est nettement moins cher. L'effet de la
localisation est donc fort mais **non linéaire** : ni la latitude ni la longitude seules ne le
résument. C'est ce qui justifiera de créer des **distances aux grandes villes** en section 9.

#### 5.3 Effets non linéaires : analyse par déciles

Pour mesurer l'effet d'une variable **quelle que soit sa forme**, je découpe chaque variable en
10 déciles et je compare la cible entre ces groupes :
- **Kruskal-Wallis** : H0 = même distribution de la cible dans tous les déciles (test sur les rangs,
  robuste aux extrêmes et au plafond) ;
- **η² (eta carré)** : la part de variance de la cible expliquée par les déciles. Contrairement à
  une corrélation, η² capte aussi une relation en U ou en cloche. Avec 20 000 observations, tout est
  significatif : c'est la **taille d'effet** qui me permet de hiérarchiser.

In [ ]:
def eta_squared(values: pd.Series, groups: pd.Series) -> float:
    # η² = variance entre groupes / variance totale (équivalent d'un R² sur une variable groupée).
    grand_mean = values.mean()
    ss_between = (
        values.groupby(groups, observed=True)
        .agg(lambda g: len(g) * (g.mean() - grand_mean) ** 2)
        .sum()
    )
    ss_total = ((values - grand_mean) ** 2).sum()
    return float(ss_between / ss_total)


rows, decile_medians = [], {}
for col in FEATURES:
    # duplicates="drop" : HouseAge a beaucoup d'ex-aequo, certains déciles fusionnent.
    deciles = pd.qcut(df_raw[col], q=10, duplicates="drop")
    groups = [y_raw[deciles == d] for d in deciles.cat.categories]
    rows.append(
        {
            "variable": col,
            "n_groupes": len(groups),
            "Kruskal-Wallis p": stats.kruskal(*groups).pvalue,
            "eta² (déciles)": eta_squared(y_raw, deciles),
            "|spearman|²": corr_target.loc[col, "spearman"] ** 2,
        }
    )
    decile_medians[col] = y_raw.groupby(deciles, observed=True).median().to_numpy()
decile_tests = (
    pd.DataFrame(rows).set_index("variable").sort_values("eta² (déciles)", ascending=False)
)
decile_tests

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for col, medians in decile_medians.items():
    ax.plot(range(1, len(medians) + 1), medians, marker="o", label=col)
ax.set(
    title="Valeur médiane de la cible par décile de chaque variable",
    xlabel="décile",
    ylabel="médiane MedHouseVal",
)
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

**Ce que j'en retiens :**
- `MedInc` reste en tête, avec une progression régulière décile après décile.
- **La comparaison entre η² et le carré de Spearman est parlante.** Pour `Latitude` et `Longitude`,
  η² est bien plus grand que ρ² : les courbes par décile montent et descendent (effet en dents de
  scie, qui correspond aux agglomérations traversées). C'est la signature d'un effet **non
  linéaire** fort qu'un modèle linéaire ne peut pas capter directement.
- `AveOccup` et `AveRooms` ont un effet réel, en partie non linéaire aussi.
- `Population` a un effet quasi nul, même par déciles.

#### 5.4 Information mutuelle

L'information mutuelle mesure la dépendance **sous toutes ses formes** entre une variable et la
cible. C'est une troisième mesure, indépendante des deux précédentes.

In [ ]:
mi = pd.Series(
    mutual_info_regression(df_raw[FEATURES], y_raw, random_state=RANDOM_STATE),
    index=FEATURES,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
mi.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title("Information mutuelle avec MedHouseVal")
plt.tight_layout()
plt.show()

### 5.5 Synthèse : les variables les plus influentes

In [ ]:
# Les trois mesures côte à côte : une variable en tête partout est un signal robuste.
influence = pd.DataFrame(
    {
        "|spearman|": corr_target["spearman"].abs(),
        "eta² (déciles)": decile_tests["eta² (déciles)"],
        "info_mutuelle": mi,
    }
)
influence["rang moyen"] = influence.rank(ascending=False).mean(axis=1)
influence.sort_values("rang moyen")

Les trois mesures convergent :
1. **`MedInc`** (revenu) est de loin la variable la plus influente ;
2. **la localisation** (`Latitude`, `Longitude`) vient ensuite, mais uniquement par ses effets non
   linéaires ;
3. **la composition des logements** (`AveOccup`, `AveRooms`) a un effet modéré ;
4. **`HouseAge`** et **`AveBedrms`** ont un effet faible ;
5. **`Population`** n'a presque aucun effet.

Je vérifierai en section 12 que le modèle final s'appuie bien sur ces variables : si les deux
analyses concordent, je peux expliquer le modèle avec confiance.

### 6. Identification des anomalies

Je distingue trois types d'anomalies, qui ne se traitent pas de la même façon :
- les **incohérences** (valeurs impossibles), que je corrige ou neutralise ;
- les **valeurs plafonnées** (vues en 2.3), que je ne peux pas corriger mais dont je tiens compte ;
- les **valeurs aberrantes** (observations réelles mais atypiques), que je ne retire que si elles
  faussent le modèle **et** qu'elles ne représentent pas ce que je veux prédire.

#### 6.1 Incohérences et districts atypiques

Je traduis le bon sens en règles vérifiables automatiquement. Je reconstitue d'abord le **nombre de
ménages** du district (`Population / AveOccup`), qui n'est pas fourni mais qui explique beaucoup
d'anomalies : un ratio calculé sur 3 ménages n'a pas la même fiabilité qu'un ratio calculé sur 500.

In [ ]:
households = df_raw["Population"] / df_raw["AveOccup"]

# Chaque règle renvoie un masque booléen : True = district concerné.
RULES = {
    "Plus de chambres que de pièces (AveBedrms > AveRooms)": df_raw["AveBedrms"]
    > df_raw["AveRooms"],
    "Moins d'une pièce par logement (AveRooms < 1)": df_raw["AveRooms"] < 1,
    "Moins d'un occupant par logement (AveOccup < 1)": df_raw["AveOccup"] < 1,
    "Très petit district (< 10 ménages)": households < 10,
    "Occupation extrême (AveOccup > 10)": df_raw["AveOccup"] > 10,
    "Logements immenses (AveRooms > 20)": df_raw["AveRooms"] > 20,
    "Cible plafonnée (MedHouseVal = max)": y_raw >= TARGET_MAX,
    "Âge plafonné (HouseAge = 52)": df_raw["HouseAge"] >= 52,
}
rules_table = pd.DataFrame(
    {
        "n_districts": {name: int(mask.sum()) for name, mask in RULES.items()},
        "% districts": {name: mask.mean() * 100 for name, mask in RULES.items()},
    }
)
rules_table

In [ ]:
# Les districts à ratios extrêmes sont-ils surtout de petits districts ?
extreme = (df_raw["AveOccup"] > 10) | (df_raw["AveRooms"] > 20)
comparison = pd.DataFrame(
    {
        "districts extrêmes": households[extreme].describe(),
        "autres districts": households[~extreme].describe(),
    }
).T[["count", "25%", "50%", "75%"]]
print("Nombre de ménages par district :")
display(comparison)
df_raw.loc[extreme].sort_values("AveOccup", ascending=False).head(8)

**Ce que j'en retiens :**
- **Aucune valeur n'est strictement impossible** : il n'y a pas de district avec plus de chambres
  que de pièces. Quelques districts ont moins d'une pièce ou d'un occupant par logement, ce qui reste
  possible (logements vacants, studios).
- **Les ratios extrêmes (environ 100 districts) viennent presque tous de très petits districts** :
  leur nombre médian de ménages est bien plus faible que celui des autres districts. Ce sont des cas
  particuliers bien connus de ce jeu de données : résidences étudiantes ou foyers (beaucoup
  d'occupants déclarés pour peu de ménages), zones de résidences secondaires ou de loisirs (beaucoup
  de pièces, peu de ménages permanents).
- Ce ne sont donc **pas des erreurs de saisie** mais des districts **atypiques et peu fiables**
  (moyennes calculées sur trop peu de ménages).

#### 6.2 Valeurs aberrantes univariées : règle de Tukey (1,5 × IQR)

L'IQR (*InterQuartile Range*, écart interquartile) est l'écart entre le 1er quartile
(Q1, 25 % des valeurs en dessous) et le 3e quartile (Q3, 75 % en dessous) : il mesure la
dispersion de la moitié centrale des données. C'est la méthode classique : une valeur est
aberrante si elle sort de
[Q1 − 1,5 × IQR ; Q3 + 1,5 × IQR].

In [ ]:
q1, q3 = df_raw[FEATURES].quantile(0.25), df_raw[FEATURES].quantile(0.75)
iqr = q3 - q1
is_out = (df_raw[FEATURES] < q1 - 1.5 * iqr) | (df_raw[FEATURES] > q3 + 1.5 * iqr)
pd.DataFrame({"n_outliers": is_out.sum(), "% outliers": is_out.mean() * 100}).sort_values(
    "n_outliers", ascending=False
)

La règle IQR signale plus de 1 000 points sur `Population`, `AveRooms`, `AveOccup` et `MedInc`.
C'est beaucoup trop pour parler d'anomalies : sur des distributions asymétriques, la règle de Tukey
(pensée pour des distributions symétriques) signale mécaniquement toute la queue de droite. Ces
districts riches ou très peuplés sont **légitimes** et doivent rester dans les données.

#### 6.3 Anomalies multivariées : Isolation Forest

L'**Isolation Forest** isole chaque point par des coupures aléatoires : les points atypiques sur
l'ensemble des variables sont isolés en peu de coupures. Je fixe `contamination=0.01` : je cherche le
1 % le plus atypique, pas à supprimer massivement.

In [ ]:
iso = IsolationForest(contamination=0.01, random_state=RANDOM_STATE)
iso_flag = pd.Series(iso.fit_predict(df_raw[FEATURES]) == -1, index=df_raw.index)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].scatter(df_raw["Longitude"], df_raw["Latitude"], c="lightgrey", s=3)
axes[0].scatter(
    df_raw.loc[iso_flag, "Longitude"],
    df_raw.loc[iso_flag, "Latitude"],
    c="crimson",
    s=12,
    label="Isolation Forest",
)
axes[0].scatter(
    df_raw.loc[extreme, "Longitude"],
    df_raw.loc[extreme, "Latitude"],
    facecolors="none",
    edgecolors="black",
    s=40,
    label="ratios extrêmes",
)
axes[0].legend()
axes[0].set_title("Localisation des districts atypiques")
sns.scatterplot(
    x=df_raw["AveOccup"],
    y=y_raw,
    hue=iso_flag,
    alpha=0.4,
    s=10,
    ax=axes[1],
    palette={False: "steelblue", True: "crimson"},
)
axes[1].set_xscale("log")
axes[1].set_title("MedHouseVal vs AveOccup (échelle log)")
plt.tight_layout()
plt.show()

print("Districts signalés par Isolation Forest :", int(iso_flag.sum()))
print("dont avec ratios extrêmes :", int((iso_flag & extreme).sum()))

**Ce que j'en retiens :** l'Isolation Forest signale environ 200 districts, en grande partie les
mêmes que la règle des ratios extrêmes, plus des districts très peuplés ou très riches. Ils sont
dispersés sur la carte (zones rurales, périphéries) : ce ne sont pas des erreurs localisées.

**Ma décision :** je ne supprime rien à ce stade. Deux options restent ouvertes, que je compare en
validation croisée en section 9 :
1. garder tous les districts ;
2. retirer de l'**entraînement uniquement** les districts à ratios extrêmes (petits districts peu
   fiables), en évaluant toujours sur **tous** les districts.

Le second point est essentiel : si je retirais aussi ces districts de l'évaluation, le score
s'améliorerait artificiellement, alors que le modèle devra quand même les prédire en production.

Ma règle est de ne supprimer une observation que si elle est atypique, que son influence sur le
modèle est démontrée, et qu'elle a une explication métier. Supprimer des points uniquement parce
qu'ils améliorent le score reviendrait à tricher avec moi-même.

### 7. Multicolinéarité

Deux variables très corrélées entre elles donnent au modèle **la même information deux fois**. Ça ne
gêne pas la prédiction, mais pour un modèle linéaire les coefficients deviennent instables et
impossibles à interpréter.

#### 7.1 Corrélations entre variables explicatives

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, method in zip(axes, ["pearson", "spearman"], strict=True):
    corr_mat = df_raw[FEATURES].corr(method=method)
    mask = np.triu(np.ones_like(corr_mat, dtype=bool))  # matrice symétrique : un seul triangle
    sns.heatmap(
        corr_mat,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        ax=ax,
        cbar_kws={"shrink": 0.7},
    )
    ax.set_title(f"Corrélations de {method.capitalize()}")
plt.tight_layout()
plt.show()

#### 7.2 Facteur d'inflation de la variance (VIF, *Variance Inflation Factor*)

Le VIF mesure à quel point une variable est expliquée par **toutes les autres** :
VIF = 1 / (1 − R²). Repères que j'utilise : **VIF > 5** = colinéarité notable, **VIF > 10** =
colinéarité problématique.

Je le calcule deux fois : sur les variables brutes, puis après un log sur les ratios très
asymétriques, parce que quelques valeurs extrêmes peuvent à elles seules gonfler ou masquer une
colinéarité.

In [ ]:
def compute_vif(df: pd.DataFrame) -> pd.Series:
    # VIF de chaque colonne. Je standardise d'abord : le VIF n'en dépend pas, le calcul est
    # simplement plus stable numériquement.
    X = (df - df.mean()) / df.std()
    X = sm.add_constant(X)  # la constante est nécessaire pour un VIF correct
    vif = [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
    return pd.Series(vif, index=df.columns, name="VIF")


X_log_ratios = df_raw[FEATURES].copy()
for col in RATIOS:
    X_log_ratios[col] = np.log(X_log_ratios[col])
pd.DataFrame(
    {
        "VIF (brut)": compute_vif(df_raw[FEATURES]),
        "VIF (log des ratios)": compute_vif(X_log_ratios),
    }
).sort_values("VIF (brut)", ascending=False)

**Ce que j'en retiens :**
- **`Latitude` et `Longitude`** sont fortement colinéaires (r ≈ −0,92, VIF ≈ 9). Ce n'est pas une
  redondance d'information mais un effet de la **forme de la Californie**, orientée en diagonale du
  nord-ouest au sud-est : quand on descend vers le sud, on va aussi vers l'est.
- **`AveRooms` et `AveBedrms`** sont très corrélés en Pearson (r ≈ 0,85, VIF ≈ 7 à 8). Mais
  après passage au log, leurs VIF retombent à 3 et 2 : l'essentiel de cette colinéarité vient de
  **quelques districts extrêmes** (6.1) où les deux ratios explosent en même temps. Pour la masse
  des districts, les deux variables restent liées (un logement avec plus de pièces a plus de
  chambres) mais sans redondance problématique.
- `MedInc` est aussi lié à `AveRooms` (les ménages aisés ont de plus grands logements).

**Mes décisions :**
1. Je privilégie des modèles **régularisés** (Ridge, Lasso, ElasticNet) pour la famille linéaire, et
   des modèles à base d'arbres, qui ne sont pas gênés par la colinéarité.
2. Plutôt que de supprimer `AveBedrms`, je le remplace en partie par un ratio **chambres / pièces**
   (section 9), qui garde l'information utile (la part de chambres dans le logement) sans être
   colinéaire avec `AveRooms`.
3. Pour la géographie, je crée des **coordonnées tournées de 45°** et des **distances aux villes**
   (section 9), qui décrivent la position autrement qu'avec deux axes colinéaires.

### 8. Approche statistique : régression linéaire (OLS) et diagnostic

Avant le machine learning, je construis un modèle linéaire classique (moindres carrés ordinaires),
interprétable. Il me sert à trois choses :
1. **quantifier** l'effet des variables (coefficients, intervalles de confiance) ;
2. **vérifier les hypothèses** de la régression linéaire : résidus normaux, variance constante,
   indépendance ;
3. mesurer l'**influence** des districts sur le modèle (distance de Cook).

C'est aussi la référence « statistique » que les modèles de machine learning devront battre.
J'applique un log à la cible et aux variables asymétriques pour me rapprocher des hypothèses du
modèle linéaire.

In [ ]:
X_ols = X_log_ratios.copy()
X_ols["MedInc"] = np.log(X_ols["MedInc"])  # le revenu agit plutôt de façon multiplicative
X_ols = sm.add_constant(X_ols)
ols = sm.OLS(y_log, X_ols).fit()
print(ols.summary())

In [ ]:
resid = ols.resid
cooks_d = ols.get_influence().cooks_distance[0]
cook_threshold = 4 / len(X_ols)  # seuil usuel : un district est influent si D > 4/n

bp_p = het_breuschpagan(resid, X_ols)[1]
diag = pd.Series(
    {
        "R² ajusté": ols.rsquared_adj,
        "Jarque-Bera p (normalité des résidus)": stats.jarque_bera(resid).pvalue,
        "Breusch-Pagan p (homoscédasticité)": bp_p,
        "Durbin-Watson (≈2 = pas d'autocorrélation)": durbin_watson(resid),
        f"n districts Cook > 4/n ({cook_threshold:.5f})": int((cooks_d > cook_threshold).sum()),
    }
)
display(diag.to_frame("valeur"))
top_cook = df_raw.assign(cook_d=cooks_d).nlargest(5, "cook_d")
display(top_cook)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(ols.fittedvalues, resid, alpha=0.1, s=5)
axes[0].axhline(0, color="crimson")
axes[0].set(title="Résidus vs valeurs ajustées", xlabel="log-valeur ajustée", ylabel="résidu")
stats.probplot(resid, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot des résidus")
# Carte des résidus : si le modèle avait tout capté, les couleurs seraient mélangées au hasard.
sc = axes[2].scatter(
    df_raw["Longitude"], df_raw["Latitude"], c=resid, cmap="RdBu_r", s=3, vmin=-0.6, vmax=0.6
)
fig.colorbar(sc, ax=axes[2], label="résidu (log)")
axes[2].set_title("Carte des résidus de l'OLS")
plt.tight_layout()
plt.show()

**Ce que j'en retiens :**
- L'OLS explique environ **66 % des écarts de valeur** entre districts (R² ajusté de 0,66) :
  c'est honorable pour 8 variables et un modèle linéaire, mais un tiers des écarts reste
  inexpliqué.
- Les coefficients se lisent facilement grâce aux logarithmes. Celui de `MedInc` (≈ 0,51) est
  une **élasticité** : **+10 % de revenu médian dans un district va avec environ +5 % de valeur
  immobilière**. `AveOccup` fait baisser la valeur (districts plus densément occupés = plus
  modestes). La latitude et la longitude sont toutes deux négatives : la valeur baisse quand on
  va vers le nord ou vers l'est, c'est-à-dire en s'éloignant de la côte sud.
- **Un signe est trompeur, et c'est l'effet direct de la multicolinéarité (7).** Seul,
  `AveRooms` est lié **positivement** à la valeur (5.1), mais dans l'OLS son coefficient est
  **négatif**, tandis que `AveBedrms` devient positif. Les deux variables se partagent la même
  information et le modèle les « oppose » : à nombre de chambres égal, plus de pièces non
  chambres… fait baisser la valeur, ce qui n'a pas de sens métier. Ces coefficients ne doivent
  donc pas être interprétés isolément.
- **Les trois hypothèses du modèle linéaire sont violées** :
  - résidus **non normaux** (Jarque-Bera), avec une queue marquée due au plafond ;
  - **hétéroscédasticité** (Breusch-Pagan) : l'erreur varie selon le niveau de valeur ;
  - **autocorrélation** : le Durbin-Watson vaut environ 1,0, loin de la valeur 2 attendue en
    l'absence d'autocorrélation. Les lignes du fichier sont rangées par zone géographique, donc des
    districts voisins se suivent, et leurs erreurs se ressemblent.
- **La carte des résidus est l'enseignement principal** : on voit des zones entières en rouge ou en
  bleu, en particulier le long de la côte. Le modèle linéaire **sous-estime systématiquement** les
  districts côtiers et surestime l'intérieur. La géographie contient une information que deux
  coefficients linéaires sur la latitude et la longitude ne peuvent pas capter.
- **Distance de Cook** : aucun district n'a une influence écrasante (la plus grande distance vaut
  environ 0,11, loin du seuil de 1 qui signale un point capable de faire basculer le modèle à lui
  seul). Les cinq districts les plus influents sont tous des districts à ratios extrêmes (6.1),
  ce qui confirme qu'ils sont atypiques sans pour autant fausser le modèle.

**Conséquences pour la suite :** il me faut des modèles capables de capter des effets
**non linéaires et géographiques** (arbres, boosting), des variables géographiques plus
parlantes, et une évaluation empirique par **validation croisée** plutôt que par les tests
paramétriques de l'OLS. L'autocorrélation spatiale m'invite aussi à vérifier en section 12 que le
modèle ne profite pas seulement de la proximité entre districts voisins.

# Phase 2 — Préparation des données

Je transforme les constats de l'EDA en décisions de prétraitement : nouvelles variables,
transformation de la cible, traitement des anomalies. Chaque décision est testée par une
expérience, et non supposée.

### 9. Feature engineering et choix de prétraitement

#### 9.1 Mise de côté du jeu de test (hold-out)

Avant de prendre la moindre décision de modélisation, je mets de côté **20 % des districts**. Je ne
les regarderai **qu'une seule fois**, à la toute fin (section 12), pour estimer honnêtement la
performance réelle. Toutes les comparaisons qui suivent (ablations, choix de modèles,
optimisation) se font uniquement par validation croisée sur les 80 % restants.

In [ ]:
X_full, y_full = df_raw[FEATURES], df_raw[TARGET]
X_train, X_hold, y_train, y_hold = train_test_split(
    X_full, y_full, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Entraînement / CV : {X_train.shape[0]} districts | Hold-out : {X_hold.shape[0]} districts")

# Plis mélangés et fixés une fois pour toutes : tous les modèles sont comparés sur exactement
# les mêmes découpages, donc les écarts de score viennent des modèles, pas du hasard.
CV = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

#### 9.2 Nouvelles variables

Chaque variable créée répond à un constat des sections précédentes :

| Nouvelle variable | Construction | Pourquoi je la crée |
|---|---|---|
| `Households` | `Population / AveOccup` | Taille réelle du district, qui explique la fiabilité des ratios (6.1) |
| `BedrmsPerRoom` | `AveBedrms / AveRooms` | Part de chambres dans le logement, non colinéaire avec `AveRooms` (7) |
| `PeoplePerRoom` | `AveOccup / AveRooms` | Densité d'occupation : un indicateur de surpeuplement |
| `IncomePerOccupant` | `MedInc / AveOccup` | Revenu rapporté à la taille du ménage |
| `Dist_<ville>` | distance à vol d'oiseau aux 5 grandes villes | L'effet géographique est centré sur les agglomérations (5.2, 8) |
| `DistNearestCity` | minimum des distances précédentes | Éloignement du pôle urbain le plus proche |
| `RotLatLon45`, `RotLatLonM45` | `Latitude ± Longitude` | Axes côte / intérieur et nord / sud, moins colinéaires que les coordonnées brutes (7) |

Toutes ces variables sont calculées ligne par ligne, **sans rien apprendre des données** : la
fonction `add_features()` peut s'appliquer telle quelle à l'entraînement, au test et en production.
Je l'intègre au pipeline avec un `FunctionTransformer`, pour que le modèle final accepte
directement les 8 colonnes brutes.

In [ ]:
def haversine_km(lat: pd.Series, lon: pd.Series, lat0: float, lon0: float) -> pd.Series:
    # Distance à vol d'oiseau (km) entre chaque district et un point de référence.
    lat, lon, lat0, lon0 = map(np.radians, (lat, lon, lat0, lon0))
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    # Feature engineering SANS ÉTAT : aucun paramètre appris, donc aucune fuite de données,
    # et le même code en entraînement et en production.
    df = df.copy()
    df["Households"] = df["Population"] / df["AveOccup"]
    df["BedrmsPerRoom"] = df["AveBedrms"] / df["AveRooms"]
    df["PeoplePerRoom"] = df["AveOccup"] / df["AveRooms"]
    df["IncomePerOccupant"] = df["MedInc"] / df["AveOccup"]
    dist_cols = []
    for city, (lat0, lon0) in CITIES.items():
        col = "Dist_" + city.replace(" ", "")
        df[col] = haversine_km(df["Latitude"], df["Longitude"], lat0, lon0)
        dist_cols.append(col)
    df["DistNearestCity"] = df[dist_cols].min(axis=1)
    df["RotLatLon45"] = df["Latitude"] + df["Longitude"]
    df["RotLatLonM45"] = df["Latitude"] - df["Longitude"]
    # Une division par zéro (ratio à 0 en production) donnerait l'infini : je le remplace par une
    # valeur manquante, que l'imputeur du pipeline saura traiter.
    return df.replace([np.inf, -np.inf], np.nan)


train_fe = add_features(X_train)
NEW_FEATURES = [c for c in train_fe.columns if c not in FEATURES]
all_corr = train_fe.corrwith(y_train, method="spearman")
new_corr = pd.DataFrame(
    {
        "spearman": all_corr[NEW_FEATURES],
        "rang parmi toutes les variables": (
            all_corr.abs().rank(ascending=False).astype(int)[NEW_FEATURES]
        ),
    }
).sort_values("spearman", key=abs, ascending=False)
new_corr

**Ce que j'en retiens :**
- **`IncomePerOccupant` devient la variable la plus corrélée à la valeur** (ρ ≈ 0,72), devant
  `MedInc` lui-même (≈ 0,68). Un revenu de 60 000 $ n'a pas le même sens pour un ménage de deux
  personnes que pour un ménage de six : rapporter le revenu à la taille du ménage donne une
  meilleure mesure du niveau de vie du quartier.
- **`RotLatLon45`** (3e, ρ ≈ −0,58) résume à elle seule l'axe **côte / intérieur** que la latitude
  et la longitude séparées ne captaient pas (leurs corrélations étaient inférieures à 0,17).
- `PeoplePerRoom` et `BedrmsPerRoom` décrivent mieux la composition des logements que les ratios
  bruts, et `DistNearestCity` confirme que la valeur baisse quand on s'éloigne des grandes villes.
- Les distances à chaque ville prises séparément sont peu corrélées : leur intérêt est ailleurs,
  pour les modèles à arbres qui peuvent les combiner (« proche de San Francisco **et** de la
  côte »). `Households` n'est quasiment pas corrélée à la cible ; elle servira surtout à repérer
  les petits districts peu fiables.

#### 9.3 Briques de modélisation

Je définis ici les fonctions réutilisées dans toute la suite. Tous les modèles suivent le même
schéma : **feature engineering → imputation → (transformation) → modèle**, le tout enveloppé dans
un `TransformedTargetRegressor` qui gère la transformation de la cible et le bornage des prédictions.

In [ ]:
def make_preprocessor(kind: str) -> Pipeline:
    # Prétraitement appris DANS le pipeline (donc refait sur chaque pli de CV, sans fuite).
    #   kind="linear" : imputation + Yeo-Johnson (corrige l'asymétrie et standardise ;
    #                   indispensable pour Ridge, Lasso, SVR, KNN, sensibles à l'échelle)
    #   kind="tree"   : imputation seule (un arbre ne dépend que de l'ordre des valeurs)
    # L'imputation est une sécurité pour la production : ce jeu n'a aucun manquant, mais une
    # donnée incomplète ou une division par zéro dans add_features() ne doit pas faire planter.
    steps = [("impute", SimpleImputer(strategy="median"))]
    if kind == "linear":
        steps.append(("power", PowerTransformer(method="yeo-johnson")))
    return Pipeline(steps)


# Fonctions de transformation de la cible. Je les définis au niveau du module (et non en lambda)
# pour que le modèle puisse être sérialisé par MLflow.
def identity(y):
    return y


def clip_target(y):
    # Borne les prédictions à l'intervalle observé : la cible est plafonnée à 5,00001 (2.3),
    # prédire au-delà ne peut qu'augmenter l'erreur.
    return np.clip(y, TARGET_MIN, TARGET_MAX)


def expm1_clip(y):
    return clip_target(np.expm1(y))


def make_model(model, kind: str, log_target: bool, clip: bool) -> TransformedTargetRegressor:
    # Pipeline complet : il prend les 8 colonnes brutes et renvoie une valeur en 100 k$.
    pipe = Pipeline(
        [
            ("features", FunctionTransformer(add_features)),
            ("prep", make_preprocessor(kind)),
            ("model", model),
        ]
    )
    if log_target:
        func, inverse = np.log1p, (expm1_clip if clip else np.expm1)
    else:
        func, inverse = identity, (clip_target if clip else identity)
    # check_inverse=False : le bornage n'est volontairement pas une fonction inverse exacte.
    return TransformedTargetRegressor(
        regressor=pipe, func=func, inverse_func=inverse, check_inverse=False
    )


def rmse(y_true, y_pred) -> float:
    return float(root_mean_squared_error(y_true, y_pred))


# Métriques calculées à chaque validation croisée. Convention scikit-learn : un score doit être
# "plus grand = meilleur", d'où les versions négatives des erreurs.
SCORING = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "mape": "neg_mean_absolute_percentage_error",
    "r2": "r2",
}

#### 9.4 Validation par ablation : chaque choix de prétraitement est testé

Plutôt que de décider sur des arguments théoriques, je teste chaque choix de prétraitement en
validation croisée, sur deux familles de modèles : **Ridge** (linéaire) et **LightGBM** (arbres).
Je fais varier une seule chose à la fois par rapport à une configuration de départ :

1. **Nouvelles variables** : avec ou sans `add_features()` ;
2. **Transformation de la cible** : cible brute ou log ;
3. **Bornage des prédictions** à l'intervalle observé [0,15 ; 5,0] ;
4. **Districts à ratios extrêmes** : gardés ou retirés de l'**entraînement** (l'évaluation se fait
   toujours sur tous les districts de chaque pli).

J'écris ma propre boucle de validation croisée pour pouvoir filtrer l'entraînement sans toucher
aux plis d'évaluation (ce que `cross_validate` ne permet pas).

In [ ]:
def is_extreme_district(X: pd.DataFrame) -> pd.Series:
    # Petits districts aux ratios peu fiables (6.1).
    return (X["AveOccup"] > 10) | (X["AveRooms"] > 20)


def cv_rmse(estimator, X, y, drop_extremes: bool = False) -> tuple[float, float]:
    # Validation croisée maison : le filtre éventuel ne s'applique qu'à la partie entraînement de
    # chaque pli ; l'évaluation porte toujours sur tous les districts du pli de validation.
    scores = []
    for tr, va in CV.split(X):
        X_tr, y_tr = X.iloc[tr], y.iloc[tr]
        if drop_extremes:
            keep = ~is_extreme_district(X_tr)
            X_tr, y_tr = X_tr[keep], y_tr[keep]
        fitted = clone(estimator).fit(X_tr, y_tr)
        scores.append(rmse(y.iloc[va], fitted.predict(X.iloc[va])))
    return float(np.mean(scores)), float(np.std(scores))


ABLATION_MODELS = {
    "Ridge": (Ridge(alpha=1.0), "linear"),
    "LightGBM": (
        LGBMRegressor(
            n_estimators=600,
            learning_rate=0.05,
            num_leaves=63,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            verbose=-1,
            n_jobs=4,
            random_state=RANDOM_STATE,
        ),
        "tree",
    ),
}
# Configuration de départ, puis une seule modification à la fois.
BASE = {"features": True, "log_target": True, "clip": True, "drop_extremes": False}
VARIANTS = {
    "0. départ (FE + log + bornage, tout gardé)": {},
    "1. sans nouvelles variables": {"features": False},
    "2. cible brute (pas de log)": {"log_target": False},
    "3. sans bornage des prédictions": {"clip": False},
    "4. districts extrêmes retirés du train": {"drop_extremes": True},
}

rows = []
for model_name, (model, kind) in ABLATION_MODELS.items():
    for variant, change in VARIANTS.items():
        cfg = {**BASE, **change}
        est = make_model(model, kind, log_target=cfg["log_target"], clip=cfg["clip"])
        if not cfg["features"]:
            # FunctionTransformer() sans fonction = identité : les 8 colonnes brutes seules.
            est.set_params(regressor__features=FunctionTransformer())
        mean, std = cv_rmse(est, X_train, y_train, drop_extremes=cfg["drop_extremes"])
        rows.append({"variante": variant, "modèle": model_name, "RMSE": mean, "écart-type": std})
ablation = pd.DataFrame(rows).pivot(
    index="variante", columns="modèle", values=["RMSE", "écart-type"]
)
ablation

**Ce que j'en retiens (chiffres de mon exécution) :**

Une RMSE de 0,43 correspond à un écart typique d'environ **43 000 $** sur la valeur médiane
d'un district ; l'écart-type entre plis est d'environ 0,01 (1 000 $). Je compare chaque variante à
la configuration de départ (ligne 0).

1. **Nouvelles variables : gain net, pour les deux familles.** Sans elles, LightGBM passe de 0,429
   à 0,445 (environ **1 600 $ d'erreur typique en plus** par district), et Ridge de 0,604 à 0,714
   (11 000 $ de plus). L'effet dépasse largement l'écart-type entre plis : le feature engineering
   est la décision de préparation la plus rentable de tout le notebook.
2. **Log de la cible : neutre pour les arbres, utile pour le linéaire.** LightGBM fait
   pratiquement la même chose dans les deux cas (0,428 contre 0,429), mais Ridge gagne nettement
   avec le log (0,604 contre 0,629, soit 2 500 $). Je garde le log, qui a en plus l'avantage de
   garantir des prédictions positives.
3. **Bornage des prédictions : petit gain, aucun coût.** Il réduit légèrement l'erreur pour les
   deux modèles (de 80 $ à 660 $), sans rien compliquer. Prédire une valeur au-delà du plafond
   de 500 000 $ ne peut de toute façon qu'augmenter l'erreur mesurée sur ces données.
4. **Retirer les districts extrêmes de l'entraînement : légère dégradation.** Les deux modèles font
   un peu moins bien (+190 $ et +120 $). Ces districts, même atypiques, apportent de l'information
   utile ; je les garde, ce qui est aussi plus simple et plus honnête pour la production.

**Mes décisions pour la suite** (je retiens un changement seulement si son effet dépasse
nettement l'écart-type entre plis, sinon je garde l'option la plus simple) :

In [ ]:
# Décisions issues de l'ablation ci-dessus.
USE_FEATURES = True
USE_LOG_TARGET = True
CLIP_PREDICTIONS = True
DROP_EXTREMES = False

print(
    f"Nouvelles variables : {USE_FEATURES} | cible en log : {USE_LOG_TARGET} | "
    f"bornage : {CLIP_PREDICTIONS} | districts extrêmes retirés du train : {DROP_EXTREMES}"
)

# Phase 3 — Modélisation

Je compare des familles de modèles très différentes à protocole égal, puis j'optimise les
meilleures.

### 10. Modélisation : comparaison des modèles avec MLflow

#### 10.1 Protocole

Pour que la comparaison soit honnête, tous les modèles passent par exactement le même protocole :
- même prétraitement (celui retenu en section 9) ;
- **validation croisée à 5 plis** sur les 80 % d'entraînement, les mêmes plis pour tous ;
- chaque modèle devient un **run MLflow** rattaché à un run parent, avec ses paramètres, ses
  métriques (moyenne et écart-type sur les plis), son score sur l'entraînement (pour détecter le
  sur-apprentissage) et son temps d'entraînement.

Je teste volontairement des familles très différentes, parce qu'on ne sait pas à l'avance laquelle
convient aux données :

| Famille | Modèles | Ce que j'en attends |
|---|---|---|
| Référence | Dummy (moyenne) | Le plancher : tout modèle utile doit faire nettement mieux |
| Linéaires | LinearRegression, Ridge, Lasso, ElasticNet | Rapides et interprétables, mais limités par la non-linéarité (8) |
| Distances | KNN (*K-Nearest Neighbors* : moyenne des k districts les plus ressemblants) | Adapté à un effet de voisinage géographique |
| Noyaux | SVR (*Support Vector Regression*, régression à vecteurs de support) | Non-linéarités douces ; coûteux sur 16 000 lignes |
| Bagging | Random Forest, Extra Trees (moyenne de nombreux arbres indépendants) | Robustes, captent les interactions |
| Boosting | GradientBoosting, HistGB (*Histogram-based Gradient Boosting*), XGBoost, LightGBM (arbres construits les uns après les autres, chacun corrigeant les erreurs des précédents) | Souvent les plus performants sur des données tabulaires |

In [ ]:
# Hyperparamètres de départ raisonnables pour ~16 000 lignes, optimisés ensuite pour les meilleurs.
# "linear" / "tree" indique le prétraitement à utiliser (voir make_preprocessor).
CANDIDATES = {
    "Dummy (moyenne)": (DummyRegressor(strategy="mean"), "linear"),
    "LinearRegression": (LinearRegression(), "linear"),
    "Ridge": (Ridge(alpha=1.0), "linear"),
    "Lasso": (Lasso(alpha=1e-4, max_iter=20_000), "linear"),
    "ElasticNet": (ElasticNet(alpha=1e-4, l1_ratio=0.5, max_iter=20_000), "linear"),
    "KNN": (KNeighborsRegressor(n_neighbors=10, weights="distance"), "linear"),
    "SVR": (SVR(C=1.0, epsilon=0.05), "linear"),
    # n_jobs=1 partout : la parallélisation se fait déjà au niveau des plis de CV
    "RandomForest": (
        RandomForestRegressor(
            n_estimators=300, min_samples_leaf=2, n_jobs=1, random_state=RANDOM_STATE
        ),
        "tree",
    ),
    "ExtraTrees": (
        ExtraTreesRegressor(
            n_estimators=300, min_samples_leaf=2, n_jobs=1, random_state=RANDOM_STATE
        ),
        "tree",
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=5,
            subsample=0.8,
            random_state=RANDOM_STATE,
        ),
        "tree",
    ),
    "HistGradientBoosting": (
        HistGradientBoostingRegressor(
            max_iter=600, learning_rate=0.08, max_leaf_nodes=63, random_state=RANDOM_STATE
        ),
        "tree",
    ),
    "XGBoost": (
        XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=7,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=1,
            random_state=RANDOM_STATE,
        ),
        "tree",
    ),
    "LightGBM": (
        LGBMRegressor(
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            n_jobs=1,
            verbose=-1,
            random_state=RANDOM_STATE,
        ),
        "tree",
    ),
}


def build(model, kind: str) -> TransformedTargetRegressor:
    # make_model avec les décisions de la section 9.
    return make_model(model, kind, log_target=USE_LOG_TARGET, clip=CLIP_PREDICTIONS)


# Si l'ablation a retenu le retrait des districts extrêmes, je l'applique à l'entraînement ici.
# (Avec cross_validate, le filtre porte alors aussi sur les plis d'évaluation : c'est sans effet
# sur la comparaison entre modèles, puisque tous sont évalués sur les mêmes districts.)
train_mask = ~is_extreme_district(X_train) if DROP_EXTREMES else pd.Series(True, X_train.index)
X_fit, y_fit = X_train[train_mask], y_train[train_mask]
print(f"Districts utilisés pour la comparaison : {len(X_fit)}")

#### 10.2 Fonctions de suivi MLflow

J'enregistre le modèle au **format MLflow** (`save_model` puis `log_artifacts`), comme dans
`components/train`. Ce format se comporte de la même façon en local et sur Azure ML, et le modèle
pourra ensuite être enregistré dans le registre par `components/register_model`.

In [ ]:
def simple_params(estimator) -> dict:
    # Hyperparamètres "simples" du modèle final, lisibles dans MLflow (les objets complexes, comme
    # un sous-estimateur, ne sont pas journalisables tels quels).
    model = (
        estimator.regressor.named_steps["model"] if hasattr(estimator, "regressor") else estimator
    )
    return {
        f"model__{k}": v
        for k, v in model.get_params(deep=False).items()
        if isinstance(v, int | float | str | bool) or v is None
    }


def log_model_artifact(estimator, X_sample: pd.DataFrame) -> None:
    # La signature décrit les colonnes attendues en entrée et le type de sortie : Azure ML s'en sert
    # pour valider les requêtes envoyées à un endpoint.
    signature = infer_signature(X_sample, estimator.predict(X_sample))
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "model"
        # MLflow 3.15 enregistre par défaut au format "skops", qui refuse les types internes de
        # scikit-learn, XGBoost et LightGBM. J'utilise donc cloudpickle. Contrepartie : un fichier
        # pickle ne doit être chargé que s'il vient d'une source de confiance (mon workspace).
        mlflow.sklearn.save_model(
            estimator,
            path,
            signature=signature,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
        mlflow.log_artifacts(str(path), artifact_path="model")


def evaluate_candidate(
    name, estimator, X, y, family: str, stage: str, log_model: bool = False
) -> dict:
    # Évalue un modèle en validation croisée et journalise tout dans un run MLflow enfant.
    with mlflow.start_run(run_name=name, nested=True):
        mlflow.set_tags(
            {
                "model_name": name,
                "family": family,
                "stage": stage,
                "log_target": USE_LOG_TARGET,
                "clip": CLIP_PREDICTIONS,
            }
        )
        mlflow.log_params(simple_params(estimator))
        start = time.time()
        # return_train_score=True : je compare le score d'entraînement au score de CV
        # pour mesurer le sur-apprentissage.
        res = cross_validate(
            estimator, X, y, cv=CV, scoring=SCORING, n_jobs=-1, return_train_score=True
        )
        metrics = {
            "cv_rmse_mean": -res["test_rmse"].mean(),
            "cv_rmse_std": res["test_rmse"].std(),
            "train_rmse_mean": -res["train_rmse"].mean(),
            "cv_mae_mean": -res["test_mae"].mean(),
            "cv_mape_mean": -res["test_mape"].mean(),
            "cv_r2_mean": res["test_r2"].mean(),
            "fit_time_mean_s": res["fit_time"].mean(),
        }
        # Écart CV − entraînement : proche de 0 = le modèle généralise ;
        # grand = il apprend par coeur.
        metrics["overfit_gap"] = metrics["cv_rmse_mean"] - metrics["train_rmse_mean"]
        # Les mêmes erreurs en milliers de dollars, pour une lecture directe côté métier.
        metrics["cv_rmse_kusd"] = metrics["cv_rmse_mean"] * TARGET_UNIT_USD / 1000
        metrics["cv_mae_kusd"] = metrics["cv_mae_mean"] * TARGET_UNIT_USD / 1000
        mlflow.log_metrics(metrics)
        if log_model:  # je n'enregistre le modèle que pour les candidats sérieux
            log_model_artifact(clone(estimator).fit(X, y), X.head(50))
        print(
            f"{name:<28} RMSE CV = {metrics['cv_rmse_mean']:.4f} ± "
            f"{metrics['cv_rmse_std']:.4f} ({time.time() - start:.0f}s)"
        )
        return {"model": name, "family": family, **metrics}

#### 10.3 Comparaison des modèles de base

Cette cellule est la plus longue du notebook (quelques minutes), à cause du SVR et des forêts
sur 16 000 districts.

In [ ]:
FAMILY = {"linear": "linéaire / noyau / distance", "tree": "arbres"}
baseline_results = []
# Un run parent regroupe les runs enfants : dans l'interface MLflow, je les retrouve ensemble.
with mlflow.start_run(run_name="01-baseline-comparison"):
    mlflow.set_tags({"stage": "baseline", "n_folds": N_FOLDS, "drop_extremes": DROP_EXTREMES})
    for name, (model, kind) in CANDIDATES.items():
        baseline_results.append(
            evaluate_candidate(name, build(model, kind), X_fit, y_fit, FAMILY[kind], "baseline")
        )

    baseline_df = pd.DataFrame(baseline_results).set_index("model").sort_values("cv_rmse_mean")
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_df = baseline_df.drop(index="Dummy (moyenne)")  # hors échelle, il écraserait le graphique
    ax.barh(plot_df.index, plot_df["cv_rmse_mean"], xerr=plot_df["cv_rmse_std"], capsize=4)
    ax.invert_yaxis()
    ax.set(title="RMSE en validation croisée (± écart-type)", xlabel="RMSE (100 k$)")
    plt.tight_layout()
    mlflow.log_figure(fig, "baseline_comparison.png")
    mlflow.log_table(baseline_df.reset_index(), "baseline_results.json")
    plt.show()

baseline_df

**Ce que j'en retiens (RMSE en CV, en centaines de milliers de dollars, chiffres de mon exécution) :**

- **La référence naïve** (prédire toujours la moyenne) se trompe typiquement de **117 000 $**
  par district : c'est le niveau d'erreur « sans modèle ».
- **Les modèles linéaires** (Ridge, Lasso, ElasticNet, LinearRegression) plafonnent tous à
  **0,60, soit environ 60 000 $** d'écart typique et un R² de 0,73. Ils sont tous équivalents :
  la régularisation n'apporte rien ici, puisqu'avec 16 000 districts et une vingtaine de
  variables il n'y a pas de sur-apprentissage à corriger. Leur limite vient de la non-linéarité
  (effet géographique, section 8), pas de la colinéarité.
- **KNN (0,53) et SVR (0,50)** font mieux en captant une partie des non-linéarités, mais restent
  loin derrière les arbres.
- **Les modèles à base d'arbres dominent**, et le **boosting** encore plus : XGBoost (0,427) et
  LightGBM (0,428) sont en tête, avec un écart typique d'environ **43 000 $**, une erreur moyenne
  (MAE) d'environ **26 000 $**, une erreur relative moyenne (MAPE) d'environ **14 %** et un
  R² de 0,86. C'est **17 000 $ d'erreur typique en moins** que les modèles linéaires.
- **Sur-apprentissage** : XGBoost a une RMSE de 0,10 sur l'entraînement contre 0,43 en CV, et le
  KNN atteint 0 sur l'entraînement (il « se souvient » de chaque district). Ces écarts sont
  attendus pour ces familles ; seul le score de CV, calculé sur des districts non vus, compte
  pour le choix.
- **Temps d'entraînement** : XGBoost, LightGBM et HistGradientBoosting s'entraînent en quelques
  secondes à quelques dizaines de secondes, alors que GradientBoosting et Random Forest demandent
  plusieurs minutes. Les valeurs exactes varient selon la charge de la machine, mais l'ordre est
  stable. C'est un critère important pour un pipeline réentraîné régulièrement.

### 11. Optimisation des meilleurs modèles

J'optimise les `TOP_K_TO_TUNE` meilleurs modèles de la section 10. J'utilise une **recherche
aléatoire** (`RandomizedSearchCV`) plutôt qu'une grille : à budget égal, elle explore mieux les
espaces à plusieurs dimensions, car elle ne perd pas de temps à tester toutes les combinaisons d'un
paramètre peu influent. Je garde les mêmes plis et le même critère (RMSE).

Pour les paramètres d'échelle (`alpha`, `learning_rate`…), je tire les valeurs selon une loi
**log-uniforme** : passer de 0,001 à 0,01 compte autant que passer de 0,1 à 1.

Je termine par un **ensemble** qui fait la moyenne des modèles optimisés : des modèles qui ne se
trompent pas exactement sur les mêmes districts compensent en partie leurs erreurs.

In [ ]:
# Préfixe imposé par l'imbrication TransformedTargetRegressor -> Pipeline -> étape "model".
P = "regressor__model__"
# Un espace de recherche par modèle. Seuls ceux qui finissent dans le top k sont utilisés.
# Le SVR n'en a pas : sur 16 000 lignes, 100 entraînements supplémentaires seraient trop longs.
SEARCH_SPACES = {
    "Ridge": {P + "alpha": loguniform(1e-3, 100)},
    "Lasso": {P + "alpha": loguniform(1e-6, 1e-2)},
    "ElasticNet": {P + "alpha": loguniform(1e-6, 1e-2), P + "l1_ratio": uniform(0.05, 0.9)},
    "KNN": {P + "n_neighbors": randint(3, 40)},
    "RandomForest": {P + "max_features": uniform(0.2, 0.7), P + "min_samples_leaf": randint(1, 8)},
    "ExtraTrees": {P + "max_features": uniform(0.2, 0.7), P + "min_samples_leaf": randint(1, 8)},
    "GradientBoosting": {
        P + "learning_rate": loguniform(0.02, 0.2),
        P + "max_depth": randint(3, 8),
        P + "subsample": uniform(0.6, 0.4),
        P + "min_samples_leaf": randint(1, 30),
    },
    "HistGradientBoosting": {
        P + "learning_rate": loguniform(0.02, 0.2),
        P + "max_leaf_nodes": randint(15, 255),
        P + "min_samples_leaf": randint(5, 60),
        P + "l2_regularization": loguniform(1e-3, 10),
        P + "max_iter": randint(300, 1500),
    },
    "XGBoost": {
        P + "learning_rate": loguniform(0.01, 0.15),
        P + "max_depth": randint(4, 11),
        P + "n_estimators": randint(500, 2500),
        P + "subsample": uniform(0.6, 0.4),
        P + "colsample_bytree": uniform(0.5, 0.5),
        P + "min_child_weight": randint(1, 20),
        P + "reg_lambda": loguniform(1e-2, 10),
    },
    "LightGBM": {
        P + "learning_rate": loguniform(0.01, 0.15),
        P + "num_leaves": randint(15, 255),
        P + "n_estimators": randint(500, 3000),
        P + "min_child_samples": randint(5, 60),
        P + "subsample": uniform(0.6, 0.4),
        P + "colsample_bytree": uniform(0.5, 0.5),
        P + "reg_lambda": loguniform(1e-2, 10),
    },
}

# Les k meilleurs modèles de la section 10 qui ont un espace de recherche.
to_tune = [m for m in baseline_df.index if m in SEARCH_SPACES][:TOP_K_TO_TUNE]
print("Modèles optimisés :", to_tune)

tuned_estimators, tuned_results, best_params = {}, [], {}
with mlflow.start_run(run_name="02-hyperparameter-tuning"):
    mlflow.set_tags({"stage": "tuning", "n_iter": N_ITER_SEARCH})
    for name in to_tune:
        model, kind = CANDIDATES[name]
        search = RandomizedSearchCV(
            build(model, kind),
            SEARCH_SPACES[name],
            n_iter=N_ITER_SEARCH,
            scoring="neg_root_mean_squared_error",
            cv=CV,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        search.fit(X_fit, y_fit)
        tuned_estimators[name] = search.best_estimator_
        best_params[name] = {k.replace(P, ""): v for k, v in search.best_params_.items()}
        # Je réévalue le meilleur réglage avec toutes les métriques, et je l'enregistre.
        tuned_results.append(
            evaluate_candidate(
                f"{name} (optimisé)",
                search.best_estimator_,
                X_fit,
                y_fit,
                FAMILY[kind],
                "tuned",
                log_model=True,
            )
        )
        print(
            "   meilleurs paramètres :",
            {k: round(v, 5) if isinstance(v, float) else v for k, v in best_params[name].items()},
        )

    # Ensemble : moyenne simple des prédictions des modèles optimisés. Je n'optimise pas les
    # poids : des poids appris sur les mêmes plis sur-apprendraient facilement.
    ensemble = VotingRegressor(list(tuned_estimators.items()))
    tuned_results.append(
        evaluate_candidate(
            "Ensemble (moyenne top-k)", ensemble, X_fit, y_fit, "ensemble", "tuned", log_model=True
        )
    )

tuned_df = pd.DataFrame(tuned_results).set_index("model").sort_values("cv_rmse_mean")
tuned_df

**Ce que j'en retiens (chiffres de mon exécution) :**

- L'optimisation apporte un **gain modeste** : XGBoost passe de 0,427 à 0,423 (environ 400 $
  d'erreur typique en moins), LightGBM de 0,428 à 0,425, GradientBoosting de 0,441 à 0,436. Les
  réglages de départ étaient déjà raisonnables.
- **L'ensemble** (moyenne des trois) obtient la meilleure RMSE, 0,4226, mais seulement 0,0005
  devant XGBoost optimisé (0,4231), soit **50 $** d'écart typique. C'est vingt fois moins que
  l'écart-type entre plis : les deux sont **à égalité statistique**.
- En revanche, l'ensemble coûte **plusieurs fois plus cher à entraîner** (il entraîne trois modèles,
  dont GradientBoosting, le plus lent) et dépend de trois librairies au lieu d'une. Je tranche en
  section 11.2.

#### 11.1 Classement général

In [ ]:
leaderboard = pd.concat([baseline_df.assign(étape="base"), tuned_df.assign(étape="optimisé")])
leaderboard.sort_values("cv_rmse_mean")[
    [
        "étape",
        "family",
        "cv_rmse_mean",
        "cv_rmse_kusd",
        "cv_rmse_std",
        "cv_mae_kusd",
        "cv_mape_mean",
        "cv_r2_mean",
        "overfit_gap",
        "fit_time_mean_s",
    ]
].head(10)

#### 11.2 Choix du modèle : la règle de l'erreur standard

Le modèle qui a la plus petite RMSE en validation croisée n'est pas forcément le bon choix : si un
modèle plus simple fait **statistiquement aussi bien**, c'est lui que je veux industrialiser. Il
sera plus rapide à réentraîner, moins cher à faire tourner et plus facile à maintenir.

J'applique la **règle de l'erreur standard** (*one-standard-error rule*) :
1. je calcule l'erreur standard de la RMSE du meilleur modèle, c'est-à-dire sa précision de
   mesure : écart-type entre plis / √(nombre de plis) ;
2. tous les modèles dont la RMSE est à moins d'une erreur standard du meilleur sont considérés
   **à égalité** ;
3. parmi eux, je retiens le **plus rapide à entraîner**.

In [ ]:
best_row = tuned_df.iloc[0]
standard_error = best_row["cv_rmse_std"] / np.sqrt(N_FOLDS)
tied = tuned_df[tuned_df["cv_rmse_mean"] <= best_row["cv_rmse_mean"] + standard_error]
print(
    f"Meilleure RMSE : {best_row['cv_rmse_mean']:.4f} ({tuned_df.index[0]}), "
    f"erreur standard : {standard_error:.4f}"
)
display(tied[["cv_rmse_mean", "cv_rmse_kusd", "fit_time_mean_s"]])

# Parmi les modèles à égalité statistique, le plus rapide à entraîner.
BEST_NAME = tied["fit_time_mean_s"].idxmin()
if BEST_NAME.startswith("Ensemble"):
    best_estimator = ensemble
else:
    best_estimator = tuned_estimators[BEST_NAME.replace(" (optimisé)", "")]
print(f"Modèle retenu : {BEST_NAME} — RMSE CV = {tuned_df.loc[BEST_NAME, 'cv_rmse_mean']:.4f}")

**Ce que j'en retiens :** l'erreur standard de la meilleure RMSE vaut environ 0,005, soit
**500 $** : c'est la précision avec laquelle je mesure l'erreur d'un modèle. L'ensemble,
XGBoost optimisé et LightGBM optimisé sont tous à moins de 500 $ les uns des autres, donc **à
égalité**. Je retiens **XGBoost optimisé** : il est plusieurs fois plus rapide à entraîner que
l'ensemble, il ne dépend que d'une librairie, et il donne exactement la même qualité
d'estimation à l'échelle d'un district (environ 42 300 $ d'écart typique contre 42 260 $).

# Phase 4 — Évaluation et conclusion

J'évalue le modèle retenu sur des données jamais vues, je vérifie sa robustesse, et je
prépare son passage dans le pipeline Azure ML.

### 12. Évaluation finale sur le hold-out

C'est le moment d'utiliser les 20 % mis de côté en section 9. J'entraîne le modèle retenu sur les
80 % d'entraînement et je l'évalue **une seule fois** sur ces districts qu'il n'a jamais vus.

Pourquoi c'est important : j'ai choisi et réglé le modèle en regardant les scores de CV, donc ces
scores sont légèrement **optimistes** (j'ai gardé ce qui marchait le mieux sur ces plis). Le hold-out,
lui, n'a influencé aucune décision : il donne une estimation honnête de la performance en production.

In [ ]:
with mlflow.start_run(run_name="03-final-holdout-evaluation") as final_run:
    # Ce tag me permettra de retrouver le modèle à industrialiser depuis le pipeline.
    mlflow.set_tags({"stage": "final", "model_name": BEST_NAME, "candidate_for_pipeline": "true"})
    if hasattr(best_estimator, "regressor"):
        mlflow.log_params(simple_params(best_estimator))
    else:
        mlflow.log_params({"ensemble_members": ", ".join(tuned_estimators)})

    best_estimator.fit(X_fit, y_fit)
    pred_hold = best_estimator.predict(X_hold)
    hold_metrics = {
        "holdout_rmse": rmse(y_hold, pred_hold),
        "holdout_mae": mean_absolute_error(y_hold, pred_hold),
        "holdout_r2": r2_score(y_hold, pred_hold),
        # MAPE : erreur relative moyenne, la plus facile à expliquer à un interlocuteur métier
        "holdout_mape_pct": mean_absolute_percentage_error(y_hold, pred_hold) * 100,
    }
    # Le plafond de la cible (2.3) : je mesure l'erreur séparément sur les districts plafonnés.
    capped = y_hold >= TARGET_MAX
    hold_metrics["holdout_rmse_non_plafonnés"] = rmse(y_hold[~capped], pred_hold[~capped])
    hold_metrics["holdout_rmse_plafonnés"] = rmse(y_hold[capped], pred_hold[capped])
    mlflow.log_metrics(hold_metrics)

    residuals = y_hold - pred_hold
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].scatter(y_hold, pred_hold, alpha=0.2, s=6)
    axes[0].plot([0, 5], [0, 5], color="crimson")  # diagonale = prédiction parfaite
    axes[0].set(title="Prédit vs réel", xlabel="réel (100 k$)", ylabel="prédit (100 k$)")
    axes[1].scatter(pred_hold, residuals, alpha=0.2, s=6)
    axes[1].axhline(0, color="crimson")
    axes[1].set(title="Résidus vs prédiction", xlabel="prédit", ylabel="réel − prédit")
    sc = axes[2].scatter(
        X_hold["Longitude"], X_hold["Latitude"], c=residuals, cmap="RdBu_r", s=5, vmin=-1, vmax=1
    )
    fig.colorbar(sc, ax=axes[2], label="résidu")
    axes[2].set_title("Carte des résidus (hold-out)")
    plt.tight_layout()
    mlflow.log_figure(fig, "holdout_residuals.png")
    plt.show()

hold_table = pd.DataFrame(
    {
        "valeur": pd.Series(hold_metrics),
        "en dollars": pd.Series(
            {
                k: f"{v * TARGET_UNIT_USD:,.0f} $"
                for k, v in hold_metrics.items()
                if "rmse" in k or "mae" in k
            }
        ),
    }
)
hold_table

**Ce que j'en retiens (chiffres de mon exécution) :**

- Sur les 4 128 districts jamais vus, XGBoost obtient une **RMSE de 0,41, soit un écart
  typique d'environ 41 000 $** sur la valeur médiane d'un district. C'est cohérent avec la
  validation croisée (0,42) : l'estimation de CV était fiable, et le modèle ne s'est pas
  sur-ajusté aux plis de sélection.
- En pratique : **en moyenne, je me trompe de 25 500 $ par district (MAE), soit 14 % de sa valeur
  (MAPE)**, et le modèle explique **87 % des écarts de valeur** entre districts (R²). Mes deux
  seuils d'acceptabilité (RMSE < 50 000 $ et MAPE < 20 %) sont respectés : le modèle est utilisable
  pour comparer et prioriser des zones.
- **L'erreur se concentre sur les districts plafonnés.** Sur les districts dont la valeur réelle est
  à 500 000 $, l'écart typique monte à **88 500 $**, contre **37 600 $** pour tous les autres. Ce
  n'est pas un défaut du modèle mais la limite des données (2.3) : la vraie valeur de ces
  districts est « 500 000 $ ou plus », inconnue. **En usage métier, une estimation proche de
  500 000 $ doit être lue comme un plancher** (« au moins 450 000 à 500 000 $ »), pas comme une
  valeur précise.
- La carte des résidus est beaucoup plus homogène que celle de l'OLS (section 8) : le modèle a
  capté l'essentiel de l'effet géographique. Il reste quelques zones sous-estimées sur la côte, là
  où se trouvent les districts plafonnés.

#### 12.1 Importance des variables (par permutation)

Pour savoir sur quoi le modèle s'appuie, je mélange une variable à la fois dans le hold-out et je
mesure de combien la RMSE se dégrade.

Un avantage du feature engineering intégré au pipeline : la permutation porte sur les **8 variables
d'origine**. Quand je mélange `Latitude`, toutes les variables qui en dérivent (distances, coordonnées
tournées) sont brouillées en même temps. J'obtiens donc la contribution **totale** de chaque
variable d'origine, sans le biais des variables dérivées redondantes qui se « couvrent » entre elles.

In [ ]:
perm = permutation_importance(
    best_estimator,
    X_hold,
    y_hold,
    scoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
perm_imp = pd.DataFrame(
    {"hausse de la RMSE": perm.importances_mean, "std": perm.importances_std}, index=FEATURES
).sort_values("hausse de la RMSE", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(perm_imp.index, perm_imp["hausse de la RMSE"], xerr=perm_imp["std"], capsize=3)
ax.invert_yaxis()
ax.set(title=f"Importance par permutation — {BEST_NAME}", xlabel="hausse de la RMSE (100 k$)")
plt.tight_layout()
with mlflow.start_run(run_id=final_run.info.run_id):  # je rattache le graphique au run final
    mlflow.log_figure(fig, "permutation_importance.png")
    mlflow.log_table(perm_imp.reset_index(names="feature"), "permutation_importance.json")
plt.show()
perm_imp

**Ce que j'en retiens :**

- **La localisation est le premier facteur de valeur pour le modèle.** Mélanger la latitude ou la
  longitude fait monter l'erreur typique d'environ **70 000 $** chacune. C'est plus que le revenu
  (`MedInc`, environ **22 000 $**), alors que le revenu était la variable la plus corrélée à la
  cible en section 5.
- Ce n'est pas une contradiction. La corrélation ne mesure que les relations linéaires, et l'effet
  géographique ne l'est pas (sections 5.2 et 8). Une fois les distances aux villes et les
  coordonnées tournées construites, le modèle exploite pleinement la localisation. L'information
  mutuelle (5.4), qui capte les relations non linéaires, plaçait d'ailleurs déjà la latitude et la
  longitude au niveau du revenu.
- Viennent ensuite la composition des logements (`AveRooms` environ 19 000 $, `AveOccup` environ
  15 000 $), puis `HouseAge` et `AveBedrms` (3 000 à 4 000 $).
- **`Population` n'apporte presque rien** (moins de 1 000 $), exactement comme l'annonçait l'analyse
  bivariée. C'est la première candidate à la suppression si je veux simplifier le modèle.

En résumé, le modèle raisonne comme on l'attend d'une estimation immobilière : **l'emplacement
d'abord, le niveau de vie ensuite, puis le type de logements**.

#### 12.2 Vérification de robustesse : validation croisée spatiale

La section 8 a montré une forte **autocorrélation spatiale** : des districts voisins se ressemblent.
Avec des plis aléatoires, un district du pli de validation a presque toujours des voisins
immédiats dans les données d'entraînement. Le modèle peut alors « tricher » en s'appuyant sur ses
voisins, ce qui rend la validation croisée optimiste.

Pour mesurer cet effet, je refais une validation croisée où les plis sont des **zones
géographiques** : je découpe la Californie en carrés de 0,5° (environ 50 km de côté) et je
garde tous les districts d'un même carré dans le même pli (`GroupKFold`). Le modèle doit alors
prédire des zones dont il n'a vu **aucun** district.

In [ ]:
geo_cells = (
    np.floor(X_fit["Latitude"] / 0.5).astype(int).astype(str)
    + "_"
    + np.floor(X_fit["Longitude"] / 0.5).astype(int).astype(str)
)
spatial = cross_validate(
    best_estimator,
    X_fit,
    y_fit,
    groups=geo_cells,
    cv=GroupKFold(n_splits=N_FOLDS),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
spatial_rmse = -spatial["test_score"].mean()
random_rmse = tuned_df.loc[BEST_NAME, "cv_rmse_mean"]
with mlflow.start_run(run_id=final_run.info.run_id):
    mlflow.log_metrics(
        {"spatial_cv_rmse_mean": spatial_rmse, "spatial_cv_rmse_std": spatial["test_score"].std()}
    )
pd.DataFrame(
    {
        "RMSE moyenne": [random_rmse, spatial_rmse],
        "écart-type": [tuned_df.loc[BEST_NAME, "cv_rmse_std"], spatial["test_score"].std()],
    },
    index=["plis aléatoires", "plis géographiques (carrés de 0,5°)"],
)

**Ce que j'en retiens :**

- Avec des plis géographiques, l'erreur typique passe de **42 000 $ à 58 000 $** (+38 %), et elle
  devient très variable d'une zone à l'autre (écart-type de 9 000 $, contre 1 000 $).
- **Ce qu'il faut en conclure dépend de l'usage :**
  - pour estimer des districts **situés dans des zones déjà couvertes** par les données (le cas
    d'usage visé : comparer les districts de Californie entre eux), l'estimation à retenir est
    celle des plis aléatoires et du hold-out, environ **41 000 à 42 000 $** d'écart typique ;
  - pour estimer une **zone entièrement nouvelle**, sans aucun district voisin dans les données,
    l'erreur attendue est d'environ **58 000 $**, au-delà de mon seuil de 50 000 $. Le modèle ne
    doit donc pas être utilisé tel quel sur une autre région sans données locales pour le
    réentraîner.
- Cette vérification montre surtout qu'une partie de la performance vient de la **connaissance du
  voisinage**. C'est une force pour le cas d'usage visé, mais c'est une limite à documenter.

#### 12.3 Modèle final : réentraînement sur toutes les données

Une fois le modèle choisi et évalué, je le réentraîne sur **100 %** des données (avec le même
filtre d'entraînement que pendant la sélection). Le hold-out a rempli son rôle d'évaluation : je
n'ai plus de raison de priver le modèle de 20 % des exemples.

J'enregistre ce modèle dans MLflow, et j'écris aussi sa **configuration complète** dans
`outputs/best_model_config.json`. Ce fichier fait le lien avec le pipeline Azure ML : ce sont ces
choix et ces hyperparamètres que le composant `components/train` reprend.

In [ ]:
full_mask = ~is_extreme_district(X_full) if DROP_EXTREMES else pd.Series(True, X_full.index)
final_model = clone(best_estimator).fit(X_full[full_mask], y_full[full_mask])


def to_builtin(value):
    # Les hyperparamètres tirés par RandomizedSearchCV sont des types numpy : je les convertis
    # en types Python pour pouvoir les écrire en JSON.
    return value.item() if hasattr(value, "item") else value


def model_params(estimator) -> dict:
    # TOUS les hyperparamètres du modèle (pas seulement ceux optimisés), pour que le composant
    # train reconstruise exactement le même modèle.
    # Les paramètres laissés à None (ou NaN) prennent la valeur par défaut de la librairie : je ne
    # les écris pas, pour garder un fichier lisible et en JSON standard.
    params = estimator.regressor.named_steps["model"].get_params(deep=False)
    return {
        k: to_builtin(v)
        for k, v in params.items()
        if isinstance(v, int | float | str | bool) and not (isinstance(v, float) and v != v)
    }


best_config = {
    "model": BEST_NAME,
    "members": {
        name: model_params(est)
        for name, est in tuned_estimators.items()
        if BEST_NAME.startswith("Ensemble") or BEST_NAME.startswith(name)
    },
    "features": FEATURES,
    "target": TARGET,
    "use_log_target": USE_LOG_TARGET,
    "clip_predictions": CLIP_PREDICTIONS,
    "target_bounds": [TARGET_MIN, TARGET_MAX],
    "drop_extreme_districts": DROP_EXTREMES,
    "cv_rmse": float(tuned_df.loc[BEST_NAME, "cv_rmse_mean"]),
    "holdout_rmse": hold_metrics["holdout_rmse"],
    "spatial_cv_rmse": float(spatial_rmse),
}
(OUTPUT_DIR / "best_model_config.json").write_text(json.dumps(best_config, indent=2))

with mlflow.start_run(run_id=final_run.info.run_id):
    log_model_artifact(final_model, X_full.head(50))
    mlflow.log_dict(best_config, "best_model_config.json")
    mlflow.set_tag("trained_on", "données complètes")

print("Run final MLflow :", final_run.info.run_id)
print(json.dumps(best_config, indent=2))

### 13. Conclusion et passage au pipeline

### Ce que je retiens de l'analyse

- **Données** : 20 640 districts californiens, 8 variables, sans valeur manquante ni doublon.
  Les vraies difficultés sont ailleurs : une **cible plafonnée** à 500 000 $ (5 % des districts),
  des **ratios extrêmes** dans une centaine de petits districts, et un **effet géographique fort
  mais non linéaire**.
- **Ce qui détermine la valeur d'un quartier** : l'emplacement (proximité de la côte et des grandes
  agglomérations), puis le niveau de vie (revenu, surtout rapporté à la taille du ménage), puis la
  composition des logements. L'âge des logements et la population du district comptent peu.
- **Préparation retenue** (chaque choix testé en validation croisée) : nouvelles variables
  géographiques et de ratios (gain net de 1 600 $ d'erreur typique pour le boosting), cible en
  log, bornage des prédictions à [15 000 $ ; 500 000 $], et **tous les districts conservés**.
- **Modèle retenu : XGBoost optimisé**, choisi par la règle de l'erreur standard parmi trois
  modèles à égalité, parce que c'est le plus rapide à entraîner.
- **Performance attendue** (hold-out) : **écart typique d'environ 41 000 $**, erreur moyenne de
  25 500 $ par district (14 % de la valeur), 87 % des écarts entre districts expliqués. C'est
  **17 000 $ d'erreur typique de moins** que le meilleur modèle linéaire, et nettement sous mes
  seuils d'acceptabilité.
- **Limites à connaître avant tout usage** :
  1. le modèle estime la **valeur typique d'un quartier**, pas le prix d'une maison précise ;
  2. au-dessus d'environ 450 000 $, une estimation doit être lue comme un **plancher** (données
     plafonnées) ;
  3. sur une **zone géographique nouvelle**, l'erreur attendue monte à environ 58 000 $ ;
  4. les données datent de **1990** : pour un usage réel, il faudrait réentraîner le pipeline sur
     des données récentes. Les montants en dollars de ce notebook sont des dollars de 1990.
- La configuration complète du modèle est exportée dans `outputs/best_model_config.json` et
  enregistrée dans le run MLflow `03-final-holdout-evaluation` (tag `candidate_for_pipeline=true`).

### Plan de portage dans le repo

| Élément du notebook | Destination dans le repo |
|---|---|
| Lecture du CSV, contrôles de schéma, découpage train / test | `components/data_prep/src/main.py` |
| `add_features()`, `make_preprocessor()`, `make_model()`, modèle retenu et ses hyperparamètres (`outputs/best_model_config.json`), transformation de la cible et bornage | `components/train/src/main.py` |
| RMSE, MAE, R², MAPE sur le jeu de test | `components/evaluate/src/main.py` |
| Enregistrement conditionnel : seuil sur la RMSE (plus bas = meilleur) | `components/register_model/src/main.py` |
| `xgboost` / `lightgbm` si le modèle retenu en dépend | `ml/environments/conda.yml` |
| `serialization_format=cloudpickle` à l'enregistrement MLflow | `components/train/src/main.py` (sinon l'enregistrement échoue avec MLflow 3.15) |
| `sample_data/california_housing.csv` | `ml/data/sample-data-asset.yml` (data asset Azure ML) |

## Annexe : check-list de la démarche

Voici la démarche suivie ici, sous forme de liste à dérouler sur un nouveau jeu de données de
régression. Seuls le dictionnaire des variables (2.2), les règles métier (6.1) et les villes de
référence (5.2) sont propres à California Housing.

**1. Cadrer**
- [ ] Définir la métrique **avant** de regarder les données, et vérifier qu'elle a un sens métier.
- [ ] Mettre en place le suivi des expériences (MLflow) dès le départ.

**2. Analyse de forme**
- [ ] Taille, types, doublons, valeurs manquantes (et ce qu'elles signifient).
- [ ] Comprendre ce que représente **une ligne** (ici un district, pas une maison).
- [ ] Chercher les valeurs **plafonnées** (censure) : pic de fréquence au maximum.

**3. Cible**
- [ ] Distribution, asymétrie, Q-Q plot, tests de normalité.
- [ ] Tester la transformation log en validation croisée plutôt que de la supposer utile.

**4 et 5. Analyse de fond**
- [ ] Univarié : asymétrie, queues extrêmes, distributions multimodales.
- [ ] Bivarié : Spearman, η² par déciles (effets non linéaires), information mutuelle.
- [ ] Données géographiques : toujours faire une **carte**.
- [ ] Hiérarchiser par **taille d'effet**, pas par p-value.

**6. Anomalies**
- [ ] Règles de cohérence métier, puis IQR, puis multivarié (Isolation Forest).
- [ ] Chercher la **cause** des anomalies (ici : les petits districts).
- [ ] Tester leur retrait en l'appliquant au **train uniquement**, jamais à l'évaluation.

**7 et 8. Multicolinéarité et modèle statistique**
- [ ] Corrélations et VIF.
- [ ] OLS de référence : vérifier les hypothèses et **cartographier les résidus**.

**9. Feature engineering**
- [ ] Mettre le hold-out de côté **avant** toute décision.
- [ ] Chaque nouvelle variable répond à un constat ; feature engineering intégré au pipeline.
- [ ] Ablation d'un seul choix à la fois ; comparer l'effet à l'écart-type entre plis.

**10 à 12. Modélisation**
- [ ] Mêmes plis pour tous les modèles ; plusieurs familles, dont une référence naïve.
- [ ] Suivre le sur-apprentissage (score d'entraînement vs score de CV).
- [ ] Optimiser les meilleurs par recherche aléatoire, puis tester un ensemble.
- [ ] Évaluer une seule fois sur le hold-out ; importance par permutation.
- [ ] Données spatiales ou temporelles : vérifier avec une validation croisée **par groupes**.
- [ ] Réentraîner sur toutes les données, enregistrer le modèle et sa configuration.